# ДЗ 5. Отток клиентов — полный проект

Единый ноутбук: разведка данных, конструирование таргета, сборка датасета, модель, uplift, экономика удержания и проверки надёжности.

**Часть 1. Данные и таргет**

| Раздел | Содержание |
|---|---|
| 0 | Подключение, конфиг, диагностика источников, качество транзакций |
| 1 | Триаж таблиц: что берём и что отбраковано |
| 2 | База наблюдений, зрелость клиентов, выбор порога тишины |
| 3 | Таргет и проверка его независимости от давности |
| 4–5 | Признаки и проверки корректности |
| 6–7 | Динамика оттока и причина роста |
| 8 | Сохранение датасета |

**Часть 2. Модель и экономика**

| Раздел | Содержание |
|---|---|
| 9–11 | Подготовка, разбиение, модель, диагностика метрики |
| 12 | Метрики, выживаемость, uplift, демография |
| 13–14 | Что влияет на отток, топ риска |
| 15 | Экономика удержания, стоимость привлечения и контакта |
| 16 | Что предлагать по топу риска |
| 17 | Проверки надёжности модели |
| 18 | Итоги для защиты |

**Как запускать.** Разделы 0–8 обращаются к базе и занимают около минуты. Разделы 9 и далее работают с датафреймом `full` в памяти, поэтому при доработке модели повторять сборку не нужно.

Три числа, задающие таргет — порог тишины, лаг и окно исхода — вынесены в конфиг раздела 0 и обоснованы в разделах 2.2 и 3.

## 0. Подключение и конфиг

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, os
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus

pd.set_option('display.max_columns', 120); pd.set_option('display.width', 220)
plt.rcParams.update({'figure.figsize': (10, 5), 'axes.grid': True, 'grid.alpha': .3})

# Пароль в файле не хранится. Берётся из переменной окружения DB_PASS,
# а если её нет — запрашивается один раз за сессию и не попадает в вывод.
DB_USER = os.getenv('DB_USER', 'amakhanov')
DB_PASS = os.getenv('DB_PASS')
if not DB_PASS:
    from getpass import getpass
    DB_PASS = getpass('Пароль к academy_db: ')
DB_HOST = os.getenv('DB_HOST', 'thomas.proxy.rlwy.net')
DB_PORT = os.getenv('DB_PORT', '51432')
DB_NAME = os.getenv('DB_NAME', 'academy_db')
DSN = os.getenv('ACADEMY_DB_URL') or \
      f'postgresql+psycopg2://{quote_plus(DB_USER)}:{quote_plus(DB_PASS)}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
engine = create_engine(DSN)
q = lambda sql, **kw: pd.read_sql(text(sql), engine, **kw)
print(q("SELECT current_database() AS db"))

In [ ]:
# ==================== КОНФИГ ====================
# Имена подставлены по фактической схеме. Ключ клиента везде customer_id,
# кроме churn_customers_crm_old (client_id) — она не используется.

SUCCESS_VALUE = 'approved'  # проверено: в churn_transactions approved/declined
USE_ARCHIVE   = False       # архив — полный дубликат основной таблицы, см. 0.3

# --- параметры таргета
# Схема: [ окно наблюдения ] t0 [ ЛАГ ] [ окно исхода ] 
#   признаки считаются строго ДО t0
#   лаг — время на скоринг, решение и контакт; в таргет не входит
#   окно исхода — период, в котором проверяется молчание
N_SILENCE = 60              # порог тишины, дней (обоснование — раздел 3.0)
LAG_DAYS  = 30              # лаг: время на решение и действие
OUTCOME_DAYS = N_SILENCE    # окно исхода равно порогу тишины по построению

SNAPSHOTS = {'S0_early': '2025-03-31',
             'S1_train': '2025-06-30',
             'S2_test':  '2025-09-30'}
DATA_END  = '2025-12-31'
MIN_TENURE_DAYS = 180       # минимальный стаж: зрелость клиента (раздел 2.1)

print('Порог тишины %d дн. | лаг %d дн. | окно исхода %d дн.'
      % (N_SILENCE, LAG_DAYS, OUTCOME_DAYS))
print()
for k, v in SNAPSHOTS.items():
    t0 = pd.Timestamp(v)
    w_start = t0 + pd.Timedelta(days=LAG_DAYS)
    w_end = w_start + pd.Timedelta(days=OUTCOME_DAYS)
    ok = 'OK' if w_end <= pd.Timestamp(DATA_END) else 'ОКНО НЕ ЗАКРЫВАЕТСЯ'
    print('%-9s t0=%s | лаг до %s | исход %s..%s  %s'
          % (k, t0.date(), w_start.date(), w_start.date(), w_end.date(), ok))

### 0.1 Диапазоны дат по всем таблицам

Таблица, у которой максимальная дата уходит дальше остальных, содержит информацию из будущего.

In [ ]:
schema_df = q("""
    SELECT table_name, column_name, data_type
    FROM information_schema.columns
    WHERE table_schema='public' AND table_name LIKE 'churn%'
    ORDER BY table_name, ordinal_position
""")

date_cols = schema_df[schema_df.data_type.str.contains('date|timestamp', case=False, na=False)]
rows = []
for _, r in date_cols.iterrows():
    try:
        d = q('SELECT min(%s) AS mn, max(%s) AS mx FROM public.%s'
              % (r.column_name, r.column_name, r.table_name))
        rows.append({'таблица': r.table_name, 'колонка': r.column_name,
                     'min': d.mn[0], 'max': d.mx[0]})
    except Exception as e:
        rows.append({'таблица': r.table_name, 'колонка': r.column_name,
                     'min': 'ошибка', 'max': str(e)[:40]})
display(pd.DataFrame(rows))

### 0.2 Статусы операций

In [ ]:
display(q("SELECT status, count(*) AS n FROM public.churn_transactions GROUP BY 1 ORDER BY 2 DESC"))
print('Подставьте фактическое значение успешной операции в SUCCESS_VALUE выше.')
print()
display(q("SELECT status, count(*) AS n FROM public.churn_subscription_charges GROUP BY 1 ORDER BY 2 DESC"))
display(q("SELECT fail_reason, count(*) AS n FROM public.churn_subscription_charges "
          "WHERE fail_reason IS NOT NULL GROUP BY 1 ORDER BY 2 DESC LIMIT 10"))

### 0.3 Что делать с churn_transactions_archive

Три возможных ответа, и выбор определяется данными:

- **Старые операции, которых нет в основной таблице** — тогда их надо приклеить, иначе история клиентов окажется обрезанной.
- **Дубликат части основной таблицы** — тогда приклеивание задвоит операции и испортит все частотные признаки.
- **Пересекающийся набор** — тогда нужен UNION с дедупликацией по `txn_id`.

In [ ]:
arch = q("""
    SELECT 'основная' AS источник, count(*) AS строк, count(DISTINCT txn_id) AS уник_txn,
           min(txn_ts)::date AS с, max(txn_ts)::date AS по
    FROM public.churn_transactions
    UNION ALL
    SELECT 'архив', count(*), count(DISTINCT txn_id),
           min(txn_ts)::date, max(txn_ts)::date
    FROM public.churn_transactions_archive
""")
display(arch)

overlap = q("""
    SELECT count(*) AS общих_txn_id
    FROM public.churn_transactions t
    JOIN public.churn_transactions_archive a USING (txn_id)
""")
display(overlap)
print('Если пересечение нулевое и периоды не совпадают — архив это старые данные,')
print('их надо приклеить (USE_ARCHIVE = True). Если пересечение велико — архив')
print('дубликат, приклеивать нельзя (USE_ARCHIVE = False).')

In [ ]:
# Источник транзакций собирается один раз и дальше используется везде
if USE_ARCHIVE is None:
    raise ValueError('Задайте USE_ARCHIVE = True или False по результатам ячейки выше')

TXN = ("""(
        SELECT txn_id, customer_id, txn_ts, txn_type, mcc_group, channel, amount_uzs, status
        FROM public.churn_transactions
        UNION
        SELECT txn_id, customer_id, txn_ts, txn_type, mcc_group, channel, amount_uzs, status
        FROM public.churn_transactions_archive
      )""" if USE_ARCHIVE else 'public.churn_transactions')

SUCC = "AND status = '%s'" % SUCCESS_VALUE
print(q('SELECT count(*) AS всего_операций FROM %s t' % TXN))

### 0.4 Качество транзакционных данных

Перед тем как строить признаки на суммах, проверяем сами суммы: отрицательные значения, подозрительно малые, валюту и распределение по каналам. Ошибка здесь испортит все денежные признаки разом.

In [ ]:
print('=== Распределение сумм ===')
display(q("""
    SELECT min(amount_uzs) AS минимум, max(amount_uzs) AS максимум,
           avg(amount_uzs)::bigint AS среднее,
           percentile_cont(0.01) WITHIN GROUP (ORDER BY amount_uzs)::bigint AS p01,
           percentile_cont(0.50) WITHIN GROUP (ORDER BY amount_uzs)::bigint AS медиана,
           percentile_cont(0.99) WITHIN GROUP (ORDER BY amount_uzs)::bigint AS p99
    FROM %s t
""" % TXN))

print('=== Аномальные суммы ===')
display(q("""
    SELECT count(*) FILTER (WHERE amount_uzs < 0)      AS отрицательные,
           count(*) FILTER (WHERE amount_uzs = 0)      AS нулевые,
           count(*) FILTER (WHERE amount_uzs > 0 AND amount_uzs < 1000) AS меньше_1000,
           count(*) FILTER (WHERE amount_uzs IS NULL)  AS пустые,
           count(*) AS всего
    FROM %s t
""" % TXN))
print('Отрицательные суммы обычно означают возвраты или сторно. Если они есть,')
print('их нельзя суммировать наравне с расходами — нужен отдельный признак.')
print()

print('=== Типы операций ===')
display(q('SELECT txn_type, count(*) AS n, avg(amount_uzs)::bigint AS средняя_сумма, '
          'min(amount_uzs) AS мин FROM %s t GROUP BY 1 ORDER BY 2 DESC' % TXN))

print('=== Каналы: онлайн против офлайн ===')
display(q('SELECT channel, count(*) AS n, count(DISTINCT customer_id) AS клиентов, '
          'avg(amount_uzs)::bigint AS средняя_сумма FROM %s t GROUP BY 1 ORDER BY 2 DESC' % TXN))
print('Если каналы делятся на онлайн и офлайн — их доля является признаком:')
print('переход клиента в онлайн или уход из него меняет паттерн активности.')
print()

print('=== Валюта ===')
cur_cols = schema_df[(schema_df.table_name.str.contains('transaction')) &
                     (schema_df.column_name.str.contains('curr', case=False))]
if len(cur_cols):
    display(cur_cols)
    print('Колонка валюты найдена — суммы нужно приводить к одной валюте.')
else:
    print('Отдельной колонки валюты нет: все суммы в одной валюте (amount_uzs — сумы).')
    print('Это надо сказать явно, иначе вопрос про мультивалютность останется открытым.')

## 1. Триаж таблиц

| Таблица | Решение | Причина |
|---|---|---|
| churn_transactions | берём | основной источник активности |
| churn_app_events | берём | поведение в приложении |
| churn_customers | берём | анкета, дата регистрации |
| churn_subscription_charges | берём | неудачные списания — сильный сигнал |
| churn_support_tickets | берём | обращения в поддержку |
| churn_subscriptions | берём с фильтром | `status` и `canceled_at` — состояние на сейчас |
| churn_products | берём с фильтром | `closed_at` — состояние на сейчас |
| churn_cards | берём с фильтром | `status` — состояние на сейчас |
| churn_nps_survey | берём | оценка лояльности |
| churn_tariff_changes | берём | вероятная причина роста оттока |
| churn_transactions_archive | по результату 0.3 | старые данные или дубликат |
| **churn_customers_360** | **не берём** | `is_churned`, `churn_flag_date`, `days_since_last_txn`, `total_txn_cnt` — посчитаны на дату выгрузки, это готовый таргет |
| **churn_customers_crm_old** | **не берём** | устаревшая копия: `client_id`, `segment_2024`, 14 000 против 20 000 |
| **churn_temp** | **не берём** | служебная, пустая |
| **churn_retention_campaigns** | **не в признаки** | вмешательство, а не наблюдение |

**Про churn_customers_360.** Это прямой аналог `scoring_applications_full` из прошлого задания. Колонка `is_churned` — тот самый таргет, который мы строим сами; `days_since_last_txn` и `total_txn_cnt` посчитаны на дату выгрузки, а не на дату снимка. Модель с ними показала бы почти идеальную метрику и не работала бы никогда.

**Про churn_retention_campaigns.** Банк связывается с теми, кого сам считает рискованными. Признак «с клиентом связывались» — следствие оценки риска, а не её причина. Используем таблицу отдельно, для оценки эффекта удержания, но не как признак модели.

## 2. База наблюдений

In [ ]:
def build_base(S):
    return q("""
        WITH last_act AS (
            SELECT customer_id, max(txn_ts)::date AS last_txn_date, count(*) AS txn_total
            FROM %(txn)s t
            WHERE txn_ts < DATE '%(S)s' %(succ)s
            GROUP BY 1
        )
        SELECT c.customer_id, c.signup_date::date AS reg_date, c.gender, c.region,
               c.segment, c.acq_channel, c.is_payroll, c.declared_income_uzs,
               c.birth_date::date AS birth_date,
               la.last_txn_date, la.txn_total,
               DATE '%(S)s' - la.last_txn_date AS days_silent,
               DATE '%(S)s' - c.signup_date::date AS tenure_days
        FROM public.churn_customers c
        LEFT JOIN last_act la ON la.customer_id = c.customer_id
    """ % dict(txn=TXN, S=S, succ=SUCC))

def apply_exclusions(base, S, label):
    n0 = len(base)
    steps = [{'шаг': 'все клиенты в базе', 'осталось': n0, 'отсечено': 0}]

    b = base[base.reg_date.notna() & (pd.to_datetime(base.reg_date) < pd.Timestamp(S))].copy()
    steps.append({'шаг': 'зарегистрированы позже снимка', 'осталось': len(b),
                  'отсечено': n0 - len(b)})

    n = len(b); b = b[b.last_txn_date.notna()]
    steps.append({'шаг': 'нет ни одной операции до снимка', 'осталось': len(b),
                  'отсечено': n - len(b)})

    n = len(b); b = b[b.days_silent < N_SILENCE]
    steps.append({'шаг': 'уже молчат %d+ дней (ушли до снимка)' % N_SILENCE,
                  'осталось': len(b), 'отсечено': n - len(b)})

    n = len(b); b = b[b.tenure_days >= MIN_TENURE_DAYS]
    steps.append({'шаг': 'незрелые: стаж менее %d дней' % MIN_TENURE_DAYS,
                  'осталось': len(b), 'отсечено': n - len(b)})

    print('=== Воронка базы наблюдений, %s (S=%s) ===' % (label, S))
    display(pd.DataFrame(steps))
    return b.reset_index(drop=True)

bases = {lb: apply_exclusions(build_base(S), S, lb) for lb, S in SNAPSHOTS.items()}

**Почему исключаем именно этих.** Клиенты без операций до снимка — признаки считать не из чего. Молчащие N дней — событие произошло до снимка, и если оставить их в обучении, модель выучит тривиальное правило «кто молчал, тот и уйдёт». Новички — недостаточно истории для поведенческих признаков.

### 2.1 Зрелые и незрелые клиенты

Клиент считается **зрелым**, если к дате снимка у него накопилось достаточно истории, чтобы поведенческие признаки имели смысл: тренды за 30/90/180 дней требуют как минимум полугода наблюдений.

Незрелые клиенты исключаются не потому, что они неинтересны, а потому что для них модель работает на пустых признаках. Их отток — отдельная задача (активация новых клиентов), и решать её нужно другими средствами.

In [ ]:
S_ref = list(SNAPSHOTS.values())[1]
mat = q("""
    SELECT c.customer_id,
           DATE '%(S)s' - c.signup_date::date AS tenure_days,
           count(t.txn_id) AS ops_before
    FROM public.churn_customers c
    LEFT JOIN %(txn)s t ON t.customer_id = c.customer_id AND t.txn_ts < DATE '%(S)s'
    WHERE c.signup_date::date < DATE '%(S)s'
    GROUP BY 1, 2
""" % dict(S=S_ref, txn=TXN))

bins = [0, 30, 90, 180, 365, 10**6]
labels = ['до 1 мес.', '1-3 мес.', '3-6 мес.', '6-12 мес.', 'больше года']
mat['группа'] = pd.cut(mat.tenure_days, bins, labels=labels)
display(mat.groupby('группа', observed=True).agg(
    клиентов=('customer_id', 'size'),
    медиана_операций=('ops_before', 'median'),
    без_операций=('ops_before', lambda s: int((s == 0).sum()))))
print()
print('Порог зрелости MIN_TENURE_DAYS = %d дней.' % MIN_TENURE_DAYS)
print('Обоснование: окна признаков доходят до 180 дней, при меньшем стаже')
print('тренды и средние считаются на неполной истории и вводят модель в заблуждение.')

### 2.2 Выбор порога тишины по кривой реактивации

Порог нельзя назначить круглым числом — его задают данные. Смотрим, какая доля клиентов возвращается после паузы длиной N дней.

Логика: пока кривая возвратов круто падает, пауза ещё может закончиться операцией — это не уход. Там, где кривая выходит на плато, возвраты становятся редкими, и молчание можно считать уходом.

Второе требование к порогу — практическое: он должен оставлять время на вмешательство. Слишком большой порог надёжен, но клиент к моменту срабатывания уже потерян.

In [ ]:
react = q("""
    WITH acts AS (
        SELECT customer_id, txn_ts::date AS d FROM %(txn)s t WHERE 1=1 %(succ)s
    ), g AS (
        SELECT customer_id, d,
               lead(d) OVER (PARTITION BY customer_id ORDER BY d) AS nxt
        FROM acts
    )
    SELECT (nxt - d) AS gap FROM g WHERE nxt IS NOT NULL
""" % dict(txn=TXN, succ=SUCC))

# Для каждого N: доля пауз, которые всё-таки закончились операцией,
# среди всех пауз длиной не меньше N (включая незавершённые).
last_act = q("""
    SELECT customer_id, max(txn_ts)::date AS last_d FROM %(txn)s t
    WHERE 1=1 %(succ)s GROUP BY 1
""" % dict(txn=TXN, succ=SUCC))
last_act['open_gap'] = (pd.Timestamp(DATA_END) - pd.to_datetime(last_act.last_d)).dt.days

rows = []
for N in [14, 21, 30, 45, 60, 75, 90, 120, 150]:
    returned = int((react.gap >= N).sum())
    still_silent = int((last_act.open_gap >= N).sum())
    total = returned + still_silent
    rows.append({'порог N': N,
                 'пауз >= N, закончились операцией': returned,
                 'молчат до конца данных': still_silent,
                 'доля вернувшихся %': round(100 * returned / max(total, 1), 1)})
rc = pd.DataFrame(rows)
display(rc)

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(rc['порог N'], rc['доля вернувшихся %'], marker='o')
ax.axvline(N_SILENCE, ls='--', color='grey', lw=1)
ax.annotate('выбранный порог: %d дн.' % N_SILENCE, xy=(N_SILENCE, rc['доля вернувшихся %'].max()),
            xytext=(6, -10), textcoords='offset points', fontsize=9, color='grey')
ax.set_xlabel('порог тишины N, дней'); ax.set_ylabel('доля вернувшихся, %')
ax.set_title('Кривая реактивации: возвращаются ли клиенты после паузы N дней')
plt.tight_layout(); plt.show()

print('Порог выбираем там, где кривая выходит на плато: дальнейшее ожидание')
print('почти не уменьшает долю ложных срабатываний, но откладывает вмешательство.')
print()
print('Второй критерий — стабильность доли оттока по месяцам при этом пороге')
print('(проверяется в разделе 6) и наличие времени на контакт (лаг %d дней).' % LAG_DAYS)

## 3. Таргет

**Определение.** Клиент считается ушедшим, если не совершил ни одной успешной операции в окне исхода — периоде длиной `N_SILENCE` дней, начинающемся через `LAG_DAYS` после даты снимка.

```
[ ---- окно наблюдения ---- ] t0 [ -- лаг -- ][ -- окно исхода -- ]
        признаки                  30 дней         60 дней
                                на действие      таргет
```

**Зачем лаг.** Без него модель предсказывает то, на что уже нельзя повлиять. Клиент, молчащий 55 дней на дату снимка, уйдёт в ближайшие дни независимо от любой кампании — предсказание верное, но бесполезное. Лаг сдвигает окно исхода вперёд и заставляет модель опираться на признаки, которые видны заранее.

**Почему это не «последнее действие».** Таргет не равен давности последней операции: он смотрит в будущее относительно `t0`, и клиент, активный на снимке, вполне может оказаться ушедшим, а давно молчащий — вернуться. Ячейка ниже печатает, сколько ушедших были активны внутри лага, то есть на дату снимка не выглядели уходящими. Отдельная проверка связи таргета с давностью — в разделе 3.1.

In [ ]:
def build_target(S):
    """Активность в ОКНЕ ИСХОДА: (t0 + LAG, t0 + LAG + OUTCOME].
    Лаг исключён из таргета — это время на скоринг, решение и контакт."""
    return q("""
        SELECT customer_id, count(*) AS txn_in_outcome,
               min(txn_ts)::date AS first_txn_outcome
        FROM %(txn)s t
        WHERE txn_ts >= DATE '%(S)s' + %(lag)d
          AND txn_ts <  DATE '%(S)s' + %(lag)d + %(out)d %(succ)s
        GROUP BY 1
    """ % dict(txn=TXN, S=S, lag=LAG_DAYS, out=OUTCOME_DAYS, succ=SUCC))

def build_lag_activity(S):
    """Активность внутри лага — в таргет не входит, нужна для диагностики."""
    return q("""
        SELECT customer_id, count(*) AS txn_in_lag
        FROM %(txn)s t
        WHERE txn_ts >= DATE '%(S)s' AND txn_ts < DATE '%(S)s' + %(lag)d %(succ)s
        GROUP BY 1
    """ % dict(txn=TXN, S=S, lag=LAG_DAYS, succ=SUCC))

targets = {}
for lb, S in SNAPSHOTS.items():
    b = bases[lb].merge(build_target(S), on='customer_id', how='left')
    b = b.merge(build_lag_activity(S), on='customer_id', how='left')
    b['txn_in_outcome'] = b.txn_in_outcome.fillna(0).astype(int)
    b['txn_in_lag'] = b.txn_in_lag.fillna(0).astype(int)

    # Отток: ни одной успешной операции в окне исхода длиной N_SILENCE дней.
    b['target'] = (b.txn_in_outcome == 0).astype(int)

    print('%s: строк %d, отток %d (%.2f%%)'
          % (lb, len(b), b.target.sum(), 100 * b.target.mean()))
    print('   из ушедших были активны внутри лага: %d (%.1f%%) — это те, кого'
          % (int(((b.target == 1) & (b.txn_in_lag > 0)).sum()),
             100 * ((b.target == 1) & (b.txn_in_lag > 0)).mean() / max(b.target.mean(), 1e-9)))
    print('   ещё можно было удержать: на дату снимка они не выглядели ушедшими')
    targets[lb] = b

print()
print('Схема: признаки до t0 | лаг %d дн. (в таргет не входит) | окно исхода %d дн.'
      % (LAG_DAYS, OUTCOME_DAYS))

### 3.1 Таргет не сводится к давности последней операции

Прямая проверка замечания «последнее действие — не таргет». Если бы отток определялся давностью, то разбиение по ней воспроизводило бы таргет один в один. Смотрим, насколько это не так.

In [ ]:
tt = targets[list(SNAPSHOTS)[1]]
tt = tt.assign(recency=tt.days_silent)

bins = [-1, 7, 14, 30, 45, N_SILENCE]
lab = ['0-7 дн.', '8-14 дн.', '15-30 дн.', '31-45 дн.', '46-%d дн.' % N_SILENCE]
tt['группа_давности'] = pd.cut(tt.recency, bins, labels=lab)

tab = tt.groupby('группа_давности', observed=True).agg(
    клиентов=('target', 'size'),
    ушли=('target', 'sum'),
    доля_ушедших_pct=('target', lambda s: round(100 * s.mean(), 2)))
display(tab)

act = tt[tt.recency <= 14]
print('Клиенты, активные в последние 14 дней перед снимком: %d, из них ушли %d (%.2f%%)'
      % (len(act), int(act.target.sum()), 100 * act.target.mean()))
print('Это те, кого давность бы не поймала: на снимке они выглядели живыми.')
print('Они составляют %.1f%% всех ушедших.'
      % (100 * act.target.sum() / max(tt.target.sum(), 1)))
print()
old_silent = tt[tt.recency > 45]
print('Клиенты, молчавшие более 45 дней: %d, из них ВЕРНУЛИСЬ %d (%.1f%%)'
      % (len(old_silent), int((old_silent.target == 0).sum()),
         100 * (old_silent.target == 0).mean()))
print('Правило «давно молчит — значит ушёл» ошиблось бы на них.')

## 4. Признаки

Каждый источник даёт свой блок. Везде жёсткое отсечение по дате снимка.

In [ ]:
def f_txn(S):
    f = q("""
        SELECT customer_id,
               max(txn_ts)::date AS last_date,
               count(*) FILTER (WHERE txn_ts >= DATE '%(S)s' -  30) AS cnt_30,
               count(*) FILTER (WHERE txn_ts >= DATE '%(S)s' -  90) AS cnt_90,
               count(*) FILTER (WHERE txn_ts >= DATE '%(S)s' - 180) AS cnt_180,
               count(*) AS cnt_all,
               sum(amount_uzs) FILTER (WHERE txn_ts >= DATE '%(S)s' -  30) AS sum_30,
               sum(amount_uzs) FILTER (WHERE txn_ts >= DATE '%(S)s' -  90) AS sum_90,
               avg(amount_uzs) FILTER (WHERE txn_ts >= DATE '%(S)s' -  90) AS avg_amt_90,
               count(DISTINCT date_trunc('month', txn_ts))
                   FILTER (WHERE txn_ts >= DATE '%(S)s' - 180) AS active_months_6m,
               count(DISTINCT mcc_group) FILTER (WHERE txn_ts >= DATE '%(S)s' - 180)
                   AS mcc_variety_6m,
               count(DISTINCT channel) FILTER (WHERE txn_ts >= DATE '%(S)s' - 180)
                   AS channel_variety_6m,
               avg(CASE WHEN channel IN ('online', 'app', 'web', 'ecom', 'internet')
                        THEN 1.0 ELSE 0.0 END)
                   FILTER (WHERE txn_ts >= DATE '%(S)s' - 90) AS online_share_90,
               avg(CASE WHEN channel IN ('online', 'app', 'web', 'ecom', 'internet')
                        THEN 1.0 ELSE 0.0 END)
                   FILTER (WHERE txn_ts >= DATE '%(S)s' - 180
                             AND txn_ts <  DATE '%(S)s' - 90) AS online_share_prev,
               sum(CASE WHEN amount_uzs < 0 THEN 1 ELSE 0 END)
                   FILTER (WHERE txn_ts >= DATE '%(S)s' - 90) AS refunds_90,
               avg(CASE WHEN status = '%(sv)s' THEN 0 ELSE 1 END)
                   FILTER (WHERE txn_ts >= DATE '%(S)s' - 90) AS decline_rate_90
        FROM %(txn)s t
        WHERE txn_ts < DATE '%(S)s'
        GROUP BY 1
    """ % dict(txn=TXN, S=S, sv=SUCCESS_VALUE))
    f['recency_days'] = (pd.Timestamp(S) - pd.to_datetime(f.last_date)).dt.days
    f['trend_30_90']  = f.cnt_30 / (f.cnt_90 / 3).replace(0, np.nan)
    f['trend_90_180'] = f.cnt_90 / (f.cnt_180 / 2).replace(0, np.nan)
    f['amt_trend_30_90'] = f.sum_30 / (f.sum_90 / 3).replace(0, np.nan)
    f['avg_gap_90'] = 90 / f.cnt_90.replace(0, np.nan)
    # сдвиг канала: клиент уходит из офлайна в онлайн или наоборот
    f['online_shift'] = f.online_share_90 - f.online_share_prev
    return f.drop(columns=['last_date'])

def f_app(S):
    f = q("""
        SELECT customer_id,
               max(event_ts)::date AS last_ev,
               count(*) FILTER (WHERE event_ts >= DATE '%(S)s' -  30) AS ev_30,
               count(*) FILTER (WHERE event_ts >= DATE '%(S)s' -  90) AS ev_90,
               count(DISTINCT date_trunc('day', event_ts))
                   FILTER (WHERE event_ts >= DATE '%(S)s' - 90) AS ev_days_90,
               count(DISTINCT event_type) FILTER (WHERE event_ts >= DATE '%(S)s' - 90)
                   AS ev_types_90,
               count(DISTINCT platform) AS platforms
        FROM public.churn_app_events
        WHERE event_ts < DATE '%(S)s'
        GROUP BY 1
    """ % dict(S=S))
    f['app_recency_days'] = (pd.Timestamp(S) - pd.to_datetime(f.last_ev)).dt.days
    f['ev_trend_30_90'] = f.ev_30 / (f.ev_90 / 3).replace(0, np.nan)
    return f.drop(columns=['last_ev'])

def f_charges(S):
    return q("""
        SELECT customer_id,
               count(*) FILTER (WHERE charged_at >= DATE '%(S)s' - 180) AS chg_180,
               count(*) FILTER (WHERE charged_at >= DATE '%(S)s' - 180
                                  AND fail_reason IS NOT NULL) AS chg_failed_180,
               count(*) FILTER (WHERE charged_at >= DATE '%(S)s' - 90
                                  AND fail_reason IS NOT NULL) AS chg_failed_90
        FROM public.churn_subscription_charges
        WHERE charged_at < DATE '%(S)s'
        GROUP BY 1
    """ % dict(S=S))

def f_tickets(S):
    f = q("""
        SELECT customer_id,
               max(opened_at)::date AS last_ticket,
               count(*) FILTER (WHERE opened_at >= DATE '%(S)s' -  90) AS tickets_90,
               count(*) FILTER (WHERE opened_at >= DATE '%(S)s' - 180) AS tickets_180,
               count(*) FILTER (WHERE opened_at >= DATE '%(S)s' - 180
                                  AND resolved_at IS NULL) AS tickets_unresolved_180,
               avg(satisfaction) FILTER (WHERE opened_at >= DATE '%(S)s' - 180)
                   AS satisfaction_avg_180
        FROM public.churn_support_tickets
        WHERE opened_at < DATE '%(S)s'
        GROUP BY 1
    """ % dict(S=S))
    f['ticket_recency_days'] = (pd.Timestamp(S) - pd.to_datetime(f.last_ticket)).dt.days
    return f.drop(columns=['last_ticket'])

def f_subs(S):
    # активной считается подписка, начатая до снимка и не отменённая на дату снимка
    return q("""
        SELECT customer_id,
               count(*) FILTER (WHERE canceled_at IS NULL OR canceled_at >= DATE '%(S)s')
                   AS subs_active,
               sum(monthly_price_uzs) FILTER (WHERE canceled_at IS NULL
                                                OR canceled_at >= DATE '%(S)s')
                   AS subs_monthly_price,
               count(*) FILTER (WHERE canceled_at >= DATE '%(S)s' - 90
                                  AND canceled_at <  DATE '%(S)s') AS subs_canceled_90
        FROM public.churn_subscriptions
        WHERE started_at < DATE '%(S)s'
        GROUP BY 1
    """ % dict(S=S))

def f_products(S):
    return q("""
        SELECT customer_id,
               count(*) FILTER (WHERE closed_at IS NULL OR closed_at >= DATE '%(S)s')
                   AS products_open,
               count(DISTINCT product_code) AS products_ever,
               count(*) FILTER (WHERE closed_at >= DATE '%(S)s' - 90
                                  AND closed_at <  DATE '%(S)s') AS products_closed_90
        FROM public.churn_products
        WHERE opened_at < DATE '%(S)s'
        GROUP BY 1
    """ % dict(S=S))

def f_cards(S):
    f = q("""
        SELECT customer_id,
               count(*) AS cards_cnt,
               min(expiry_date)::date AS nearest_expiry
        FROM public.churn_cards
        WHERE issue_date < DATE '%(S)s'
        GROUP BY 1
    """ % dict(S=S))
    f['days_to_expiry'] = (pd.to_datetime(f.nearest_expiry) - pd.Timestamp(S)).dt.days
    return f.drop(columns=['nearest_expiry'])

def f_nps(S):
    f = q("""
        SELECT DISTINCT ON (customer_id)
               customer_id, nps_score AS nps_last, survey_date::date AS nps_date
        FROM public.churn_nps_survey
        WHERE survey_date < DATE '%(S)s'
        ORDER BY customer_id, survey_date DESC
    """ % dict(S=S))
    f['nps_recency_days'] = (pd.Timestamp(S) - pd.to_datetime(f.nps_date)).dt.days
    return f.drop(columns=['nps_date'])

def f_tariff(S):
    f = q("""
        SELECT customer_id,
               max(effective_date)::date AS last_change,
               count(*) AS tariff_changes_cnt,
               max(new_monthly_fee_uzs - old_monthly_fee_uzs) AS fee_delta_max
        FROM public.churn_tariff_changes
        WHERE effective_date < DATE '%(S)s'
        GROUP BY 1
    """ % dict(S=S))
    f['days_since_tariff_change'] = (pd.Timestamp(S) - pd.to_datetime(f.last_change)).dt.days
    f['tariff_changed_90d'] = (f.days_since_tariff_change <= 90).astype(int)
    return f.drop(columns=['last_change'])

# f_nps исключён: опросы проводились с 2025-11-01 по 2025-12-31, то есть ПОСЛЕ обоих
# снимков. Ключ и тип данных корректны, но период не покрывает момент предсказания —
# использование признака означало бы заглядывание в будущее.
BUILDERS = [f_txn, f_app, f_charges, f_tickets, f_subs, f_products, f_cards, f_tariff]

In [ ]:
datasets = {}
for lb, S in SNAPSHOTS.items():
    d = targets[lb].copy()
    for fn in BUILDERS:
        try:
            d = d.merge(fn(S), on='customer_id', how='left')
        except Exception as e:
            print('  [!] %s пропущен: %s' % (fn.__name__, str(e)[:90]))
    d['age_years'] = ((pd.Timestamp(S) - pd.to_datetime(d.birth_date)).dt.days / 365.25).round(1)
    d['snapshot'] = S
    d['snapshot_label'] = lb
    datasets[lb] = d
    print('%s: %d строк, %d колонок, отток %.2f%%' % (lb, len(d), d.shape[1], 100 * d.target.mean()))

full = pd.concat(datasets.values(), ignore_index=True)
print()
print('Итого %d строк, %d уникальных клиентов' % (len(full), full.customer_id.nunique()))

## 5. Проверки корректности

In [ ]:
print('=== 1. Утечка из будущего ===')
print('строк с отрицательной давностью операций: %d (должно быть 0)'
      % int((full.recency_days < 0).sum()))
if 'app_recency_days' in full:
    print('строк с отрицательной давностью событий:  %d (должно быть 0)'
          % int((full.app_recency_days < 0).sum()))

print()
print('=== 2. Пересечение клиентов между снимками ===')
lbs = list(SNAPSHOTS)
a, b = set(datasets[lbs[0]].customer_id), set(datasets[lbs[1]].customer_id)
print('в обоих снимках: %d клиентов (%.1f%% проверочного)'
      % (len(a & b), 100 * len(a & b) / max(len(b), 1)))
print('Панельная схема — пересечение ожидаемо. Но метрику стоит посчитать')
print('дополнительно только на клиентах, которых не было в обучении.')

print()
print('=== 3. Доля оттока по снимкам ===')
display(full.groupby('snapshot_label').agg(
    строк=('target', 'size'), отток=('target', 'sum'),
    доля_pct=('target', lambda s: round(100 * s.mean(), 2))))

print()
print('=== 4. Пропуски ===')
SERVICE = ['customer_id', 'target', 'snapshot', 'snapshot_label', 'reg_date', 'birth_date',
           'last_txn_date', 'txn_in_outcome', 'first_txn_outcome', 'txn_in_lag', 'silence_at_end',
           'days_silent', 'txn_total', 'full_name']
feat_cols = [c for c in full.columns if c not in SERVICE]
na = (full[feat_cols].isna().mean() * 100).round(1).sort_values(ascending=False)
display(na[na > 0].to_frame('пропусков_%'))
print('Высокая доля пропусков нормальна для источников, которые есть не у всех')
print('(NPS, тикеты, подписки). Заполняем нулём там, где пропуск означает «не было события».')

### 5.1 Диагностика NPS

Признак `nps_last` пуст на 100% строк — это не «данных нет у части клиентов», а сбой соединения. В таблице 4 600 строк, значит либо опросы проводились позже снимков, либо ключ не совпадает, либо дата хранится текстом.

In [ ]:
print('=== Период опросов ===')
display(q("SELECT min(survey_date)::date AS с, max(survey_date)::date AS по, "
          "count(*) AS строк, count(DISTINCT customer_id) AS клиентов "
          "FROM public.churn_nps_survey"))

print('=== Сколько опросов доступно до каждого снимка ===')
for lb, S in SNAPSHOTS.items():
    n = q("SELECT count(*) AS n FROM public.churn_nps_survey WHERE survey_date < DATE '%s'" % S).n[0]
    print('  %-9s S=%s : %d опросов' % (lb, S, n))

print('=== Совпадает ли ключ с таблицей клиентов ===')
display(q("""SELECT count(*) AS строк_опроса,
                    count(c.customer_id) AS нашлось_клиентов
             FROM public.churn_nps_survey n
             LEFT JOIN public.churn_customers c USING (customer_id)"""))

print('=== Тип колонки ===')
display(schema_df[(schema_df.table_name == 'churn_nps_survey')])
print()
print('Если опросов до снимка ноль — признак убираем из BUILDERS и говорим об этом')
print('в защите: источник есть, но он не покрывает нужный период.')

### 5.2 Заполнение пропусков

Заполнить всё нулями нельзя: смысл пропуска у разных признаков разный.

- **Счётчики событий** — пропуск означает «событий не было», это честный ноль.
- **Давность события** — ноль означал бы «событие было сегодня», то есть противоположность правде. Ставим заведомо большое значение и добавляем флаг наличия.
- **Средние и оценки** — `satisfaction = 0` означало бы «оценил на ноль». Оставляем пропуском, его заполнит препроцессор медианой, а флаг наличия несёт собственный сигнал.

Флаги `has_*` заслуживают отдельного внимания: в скоринге два из трёх сильнейших признаков оказались именно такими — сам факт наличия информации о клиенте информативнее её содержания.

In [ ]:
# Защита: колонки, пустые на 100%, не несут информации и мешают препроцессору
all_nan = [c for c in full.columns if full[c].isna().all()]
if all_nan:
    print('удалены полностью пустые колонки:', all_nan)
    full = full.drop(columns=all_nan)
else:
    print('полностью пустых колонок нет')

COUNT_PREFIX = ('cnt_', 'ev_', 'sum_', 'tickets_', 'chg_', 'subs_', 'products_',
                'cards_cnt', 'tariff_changes_cnt', 'mcc_variety', 'channel_variety',
                'active_months', 'ev_days', 'ev_types', 'platforms')
COUNT_COLS = [c for c in full.columns
              if any(c.startswith(p) for p in COUNT_PREFIX) and c not in ('ev_trend_30_90',)]
TREND_COLS = [c for c in full.columns if 'trend' in c]
RECENCY_COLS = [c for c in full.columns if c.endswith('_recency_days')]
KEEP_NAN = [c for c in ['satisfaction_avg_180', 'nps_last', 'avg_amt_90', 'decline_rate_90',
                        'days_to_expiry', 'avg_gap_90', 'subs_monthly_price', 'fee_delta_max']
            if c in full.columns]

full[COUNT_COLS] = full[COUNT_COLS].fillna(0)
full[TREND_COLS] = full[TREND_COLS].fillna(0)
if 'tariff_changed_90d' in full:
    full['tariff_changed_90d'] = full.tariff_changed_90d.fillna(0)

for c in RECENCY_COLS:
    full['has' + c.replace('_recency_days', '')] = full[c].notna().astype(int)
    full[c] = full[c].fillna(9999)

for c in KEEP_NAN:
    full['has_' + c] = full[c].notna().astype(int)

print('заполнено нулями        : %d признаков' % (len(COUNT_COLS) + len(TREND_COLS)))
print('давность + флаг наличия : %d' % len(RECENCY_COLS))
print('оставлено пропуском     : %d' % len(KEEP_NAN))
print('флагов has_* добавлено  : %d' % sum(c.startswith('has') for c in full.columns))
print()
feat_cols = [c for c in full.columns if c not in SERVICE]
na2 = (full[feat_cols].isna().mean() * 100).round(1)
print('Признаков с пропусками осталось: %d (это ожидаемо, см. KEEP_NAN)'
      % int((na2 > 0).sum()))
display(na2[na2 > 0].to_frame('пропусков_%'))

### 5.3 Пересечение клиентов: контрольная выборка

Пересечение 90% между снимками — не формальность. В скоринге устранение 40-процентного пересечения стоило 0,09 Gini.

Опасность: клиент, активный в июне, скорее всего активен и в сентябре. Модель может выучить не признаки-предвестники ухода, а стабильность конкретных людей — на новых клиентах такое знание бесполезно.

Полностью убрать пересечение нельзя: в проверке останется около 1 200 строк. Поэтому на этапе моделирования метрика считается на обеих выборках.

**Важная оговорка при трактовке.** «Новые» клиенты здесь — не случайная подвыборка, а другая популяция: те, кто в июньский снимок не попал (зарегистрировались позже, не имели операций или уже молчали). Их доля оттока 9,38% против 5,77% на полном снимке, то есть сегмент объективно более рискованный и хуже предсказуемый. Поэтому более низкая метрика на нём **не доказывает** запоминание людей — она может отражать разницу популяций. Сравнение остаётся полезным, но вывод формулируется осторожно: если метрика держится, пересечение точно не вредит; если падает, нужна дополнительная проверка.

In [ ]:
lbs = list(SNAPSHOTS)
lb_last = lbs[-1]

# «Новый» — клиент, которого не было ни в одном из предыдущих снимков,
# то есть модель его действительно не видела при обучении.
seen_before = set()
for lb in lbs[:-1]:
    seen_before |= set(datasets[lb].customer_id)

full['is_new_client'] = (~full.customer_id.isin(seen_before)).astype(int)

test_mask = full.snapshot_label == lb_last
new_test = full[test_mask & (full.is_new_client == 1)]

print('Снимков всего: %d, проверочный — %s' % (len(lbs), lb_last))
print('Проверочный снимок целиком : %d строк, отток %.2f%%'
      % (int(test_mask.sum()), 100 * full[test_mask].target.mean()))
print('Только новые клиенты       : %d строк, отток %.2f%%'
      % (len(new_test), 100 * new_test.target.mean() if len(new_test) else 0))
print()
print('На этапе моделирования метрика считается на обеих выборках.')
print('Колонка is_new_client сохранена в датасете.')

## 6. Динамика оттока по месяцам

Считается отдельно от датасета — по когортам клиентов, замолчавших в каждом месяце.

In [ ]:
monthly = q("""
    WITH acts AS (
        SELECT customer_id, txn_ts::date AS d FROM %(txn)s t WHERE 1=1 %(succ)s
    ), g AS (
        SELECT customer_id, d, lead(d) OVER (PARTITION BY customer_id ORDER BY d) AS nxt
        FROM acts
    )
    SELECT date_trunc('month', d)::date AS месяц,
           count(*) FILTER (WHERE nxt IS NULL OR nxt - d >= %(N)d) AS ушли,
           count(DISTINCT customer_id) AS активных
    FROM g GROUP BY 1 ORDER BY 1
""" % dict(txn=TXN, succ=SUCC, N=N_SILENCE))
monthly['доля_оттока_pct'] = (100 * monthly.ушли / monthly.активных).round(2)

# Последние N дней читать нельзя: у клиентов не было возможности намолчать
# полный порог. Отрезаем цензурированный хвост, иначе он ломает и график, и выводы.
CENSOR_FROM = pd.Timestamp(DATA_END) - pd.Timedelta(days=N_SILENCE)
readable = monthly[pd.to_datetime(monthly.месяц) <= CENSOR_FROM].copy()

print('Полный ряд (последние месяцы цензурированы):')
display(monthly.tail(6))
print('Цензурирование начинается с %s — эти месяцы из выводов исключаем.'
      % CENSOR_FROM.date())
print()

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(readable.месяц, readable.доля_оттока_pct, marker='o')
base = readable[pd.to_datetime(readable.месяц) < '2025-06-01'].доля_оттока_pct.mean()
ax.axhline(base, ls='--', color='grey', lw=1)
ax.annotate('средний уровень до июня 2025: %.2f%%' % base,
            xy=(readable.месяц.iloc[1], base), xytext=(0, 8),
            textcoords='offset points', fontsize=9, color='grey')
ax.set_title('Доля оттока по месяцам, %% (цензурированный хвост убран)')
ax.set_ylabel('%')
plt.tight_layout(); plt.show()

peak = readable.loc[readable.доля_оттока_pct.idxmax()]
print('Максимум читаемой части: %s — %.2f%% (базовый уровень %.2f%%, рост в %.1f раза)'
      % (peak.месяц, peak.доля_оттока_pct, base, peak.доля_оттока_pct / base))

## 7. Смена тарифа как причина роста

Совпадение по времени — гипотеза, а не доказательство. Проверяем на разнице между затронутыми и незатронутыми клиентами.

In [ ]:
# 7.1 Когда менялись тарифы и на сколько
tar_m = q("""
    SELECT date_trunc('month', effective_date)::date AS месяц,
           count(*) AS изменений,
           count(DISTINCT customer_id) AS клиентов,
           round(avg(new_monthly_fee_uzs - old_monthly_fee_uzs)) AS средняя_дельта,
           count(*) FILTER (WHERE new_monthly_fee_uzs > old_monthly_fee_uzs) AS повышений
    FROM public.churn_tariff_changes
    GROUP BY 1 ORDER BY 1
""")
display(tar_m)

display(q("""SELECT reason, count(*) AS n,
                    round(avg(new_monthly_fee_uzs - old_monthly_fee_uzs)) AS средняя_дельта
             FROM public.churn_tariff_changes GROUP BY 1 ORDER BY 2 DESC"""))

# 7.2 Совпадает ли всплеск изменений с всплеском оттока
joined = readable.merge(tar_m[['месяц', 'клиентов']], on='месяц', how='left')
joined['клиентов'] = joined.клиентов.fillna(0)
fig, ax1 = plt.subplots(figsize=(11, 4.5))
ax1.plot(joined.месяц, joined.доля_оттока_pct, marker='o', label='отток, %')
ax1.set_ylabel('отток, %')
ax2 = ax1.twinx(); ax2.grid(False)
ax2.bar(joined.месяц, joined.клиентов, width=20, alpha=.3, color='grey',
        label='клиентов со сменой тарифа')
ax2.set_ylabel('клиентов со сменой тарифа')
ax1.set_title('Отток и смена тарифа по месяцам')
plt.tight_layout(); plt.show()

In [ ]:
# 7.3 Разница в оттоке между затронутыми и незатронутыми
grp = full.tariff_changed_90d.fillna(0).astype(int)
cmp_tar = full.groupby(grp).agg(
    клиентов=('target', 'size'),
    отток_pct=('target', lambda s: round(100 * s.mean(), 2)))
LABELS = {0: 'тариф не менялся за 90 дн. до снимка', 1: 'тариф менялся за 90 дн. до снимка'}
cmp_tar.index = [LABELS.get(i, str(i)) for i in cmp_tar.index]
display(cmp_tar)

if cmp_tar.shape[0] < 2:
    print('[!] Затронутых клиентов в окне 90 дней нет — окно слишком узкое.')
    print('    Смотрим более широкие окна ниже.')

# более широкие окна: смена тарифа могла произойти задолго до снимка
d = full.days_since_tariff_change
buckets = pd.cut(d, [-1, 90, 180, 365, 10**6],
                 labels=['≤90 дн.', '91–180 дн.', '181–365 дн.', '>365 дн.'])
bt = full.assign(bucket=buckets).groupby('bucket', observed=True).agg(
    клиентов=('target', 'size'),
    отток_pct=('target', lambda s: round(100 * s.mean(), 2)))
never = full[d.isna()]
bt.loc['тариф не менялся никогда'] = [len(never), round(100 * never.target.mean(), 2)]
display(bt)

# 7.4 Повышение платы против остальных изменений
if 'fee_delta_max' in full:
    up = full[full.fee_delta_max > 0]
    same = full[(full.fee_delta_max <= 0) & full.fee_delta_max.notna()]
    print('повышение платы : %5d клиентов, отток %.2f%%'
          % (len(up), 100 * up.target.mean() if len(up) else 0))
    print('без повышения   : %5d клиентов, отток %.2f%%'
          % (len(same), 100 * same.target.mean() if len(same) else 0))
    print('тариф не менялся: %5d клиентов, отток %.2f%%'
          % (len(never), 100 * never.target.mean()))
print()
print('Совпадение по времени — гипотеза, а не доказательство. Разница в оттоке')
print('между затронутыми и незатронутыми — уже аргумент. Но и его надо проверить:')
print('тариф могли менять не случайной выборке, а определённому сегменту.')

In [ ]:
# 7.5 Корректное сравнение: ВНУТРИ одного снимка
# Тарифы изменены 2025-07-01, то есть после S1 (30.06) и за 91 день до S2 (30.09).
# Значит затронутые есть только в сентябрьском снимке, а группа «не менялся»
# в таблице выше включала оба снимка — сравнивались разные популяции.
lb_test = list(SNAPSHOTS)[-1]   # последний снимок: только он лежит после реформы
s2 = full[full.snapshot_label == lb_test].copy()
s2['affected'] = s2.days_since_tariff_change.notna().astype(int)

cmp2 = s2.groupby('affected').agg(
    клиентов=('target', 'size'),
    ушли=('target', 'sum'),
    отток_pct=('target', lambda s: round(100 * s.mean(), 2)))
LBL = {0: 'тариф не менялся', 1: 'тариф повышен 01.07.2025'}
cmp2.index = [LBL.get(i, str(i)) for i in cmp2.index]
display(cmp2)

if cmp2.shape[0] < 2:
    print('[!] В снимке %s только одна группа — сравнение невозможно.' % lb_test)
    print('    Тарифы изменены 01.07.2025, значит затронутые есть только')
    print('    в снимках ПОСЛЕ этой даты.')
    raise SystemExit

# z-тест разности долей
from math import sqrt
n1, x1 = int(cmp2.клиентов.iloc[0]), int(cmp2.ушли.iloc[0])
n2, x2 = int(cmp2.клиентов.iloc[1]), int(cmp2.ушли.iloc[1])
p1, p2 = x1 / n1, x2 / n2
pp = (x1 + x2) / (n1 + n2)
z = (p2 - p1) / sqrt(pp * (1 - pp) * (1 / n1 + 1 / n2))
se = sqrt(p1 * (1 - p1) / n1 + p2 * (1 - p2) / n2)
lo, hi = (p2 - p1) - 1.96 * se, (p2 - p1) + 1.96 * se
print('Разность долей: %+.2f п.п., z = %.2f' % (100 * (p2 - p1), z))
print('95%% интервал разности: [%+.2f; %+.2f] п.п.' % (100 * lo, 100 * hi))
print('Значимо' if lo > 0 or hi < 0 else 'Незначимо: интервал накрывает ноль')
print()

# 7.6 Сопоставимы ли группы
# Если тариф повышали не случайной выборке, разница в оттоке может объясняться
# составом групп, а не самим повышением.
print('=== Состав групп ===')
for col in ['segment', 'region', 'acq_channel', 'is_payroll']:
    if col in s2.columns:
        t = pd.crosstab(s2[col], s2.affected, normalize='columns') * 100
        t.columns = ['не менялся, %', 'повышен, %']
        display(t.round(1))

print('=== Поведение до снимка ===')
display(s2.groupby('affected')[['tenure_days', 'cnt_90', 'recency_days', 'age_years']]
        .median().round(1).rename(index={0: 'не менялся', 1: 'повышен'}))
print()
print('Если группы близки по составу и поведению — повышение тарифа')
print('можно считать причиной. Если различаются — эффект смешан с сегментом,')
print('и корректнее сравнивать внутри однородных подгрупп.')

## 8. Сохранение

In [ ]:
try:
    full.to_parquet('churn_dataset.parquet', index=False)
    print('Сохранено: churn_dataset.parquet — %d строк, %d колонок' % full.shape)
except ImportError:
    full.to_csv('churn_dataset.csv', index=False, encoding='utf-8')
    print('pyarrow не установлен, сохранено в CSV: churn_dataset.csv — %d строк, %d колонок'
          % full.shape)
    print('Для parquet: pip install pyarrow')

feat_cols = [c for c in full.columns if c not in SERVICE]
print('Признаков: %d' % len(feat_cols))
print(feat_cols)
display(full.head(10))

Датасет собран и сохранён. Вторая часть работает с переменной `full` напрямую, без повторного обращения к базе.

---
# Часть 2. Модель

Датасет собран выше и лежит в переменной `full`. Дальше — обучение, оценка качества и экономика удержания.

Ключевые решения этой части:

1. **Разбиение по снимку**, а не случайное: обучение на раннем срезе, проверка на следующем.
2. **Тарифные признаки исключаются** — их не существует в обучающем срезе, реформа произошла 1 июля.
3. **Метрика ранжирования**, а не точности: бизнесу нужен список приоритетов, а не вердикт по каждому.
4. **Экономика задаёт глубину обработки**: стоимость контакта против ценности клиента.

## 9. Подготовка к моделированию

In [ ]:
# Дополнительные импорты для моделирования.
# Подключение, pandas/numpy/matplotlib и датафрейм full уже созданы в части 1.
import importlib, subprocess, sys

if importlib.util.find_spec('sklearn') is None:
    print('scikit-learn не найден, устанавливаю...')
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'scikit-learn'],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-1200:])
        raise SystemExit('Установка не удалась. Переключите ядро на Python 3.11-3.12.')
    importlib.invalidate_caches()
    print('Установлено. Если импорты ниже падают — перезапустите ядро.')

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, brier_score_loss

print('Датасет из части 1: %d строк, %d колонок' % full.shape)
display(full.groupby('snapshot_label').target.agg(['size', 'sum', 'mean']).round(4))

## 10. Разбиение и отбор признаков

Обучение на раннем снимке, проверка на следующем. Случайное разбиение здесь недопустимо: оно поместило бы в обучение информацию, которой в момент предсказания не существовало.

In [ ]:
SERVICE = ['customer_id', 'target', 'snapshot', 'snapshot_label', 'reg_date', 'birth_date',
           'last_txn_date', 'txn_in_outcome', 'first_txn_outcome', 'txn_in_lag', 'silence_at_end',
           'days_silent', 'txn_total', 'full_name', 'is_new_client']

train = full[full.snapshot_label == 'S1_train'].copy()
test  = full[full.snapshot_label == 'S2_test'].copy()
cand  = [c for c in full.columns if c not in SERVICE]

print('обучение: %d строк, отток %.2f%%' % (len(train), 100 * train.target.mean()))
print('проверка: %d строк, отток %.2f%%' % (len(test), 100 * test.target.mean()))

### 10.1 Отсев непригодных признаков

Две автоматические проверки, каждая ловит конкретную проблему.

**Нулевая дисперсия в обучении.** Признак, принимающий одно значение на всех обучающих строках, не несёт информации — модель не может на нём ничему научиться. Здесь так себя ведут тарифные признаки: реформа состоялась 1 июля, то есть после обучающего снимка.

**Расхождение покрытия между снимками.** Если признак заполнен у 5% обучающих строк и у 40% проверочных, модель встретит в проверке распределение, которого не видела. Это не утечка, но источник нестабильности, и такие признаки надо находить осознанно, а не обнаруживать по странной метрике.

In [ ]:
rows = []
for col in cand:
    tr, te = train[col], test[col]
    nun = tr.nunique(dropna=True)
    rows.append({
        'признак': col,
        'уник_в_train': nun,
        'заполнено_train_%': round(100 * tr.notna().mean(), 1),
        'заполнено_test_%': round(100 * te.notna().mean(), 1),
    })
diag = pd.DataFrame(rows)
diag['разрыв_покрытия'] = (diag['заполнено_test_%'] - diag['заполнено_train_%']).round(1)

zero_var = diag[diag.уник_в_train <= 1].признак.tolist()
big_gap  = diag[(diag.разрыв_покрытия.abs() > 20) & (~diag.признак.isin(zero_var))].признак.tolist()

print('Признаки с нулевой дисперсией в обучении (исключаем):')
print(' ', zero_var if zero_var else 'нет')
print()
print('Признаки с большим разрывом покрытия между снимками:')
display(diag[diag.признак.isin(big_gap)].sort_values('разрыв_покрытия'))

DROP = set(zero_var) | set(big_gap)
FEATS = [c for c in cand if c not in DROP]
print()
print('Признаков осталось: %d из %d' % (len(FEATS), len(cand)))

**Почему тарифные признаки исключены.** Реформа вступила в силу 1 июля 2025, обучающий снимок — 30 июня. В обучении о ней нет ни одной записи, в проверке она затрагивает 39% клиентов и объясняет заметную часть оттока.

Модель, обученная на июне, физически не может научиться реагировать на это событие. Оставить признаки — значит создать ложное впечатление, что модель их учла.

Это содержательный вывод, а не техническая неудача: **разовый шок вроде тарифной реформы моделью не покрывается**. Он требует отдельного сценарного расчёта и переобучения модели после события. Эффект реформы измерен отдельно в разделе 7 предыдущего ноутбука: +4,19 п.п. к оттоку, при статистически неразличимом составе групп.

## 11. Модель

In [ ]:
def build(cols, kind='logreg', C=1.0):
    num = [c for c in cols if pd.api.types.is_numeric_dtype(train[c])]
    cat = [c for c in cols if c not in num]
    steps = [('num', Pipeline([('i', SimpleImputer(strategy='median')),
                               ('s', StandardScaler())]), num)]
    if cat:
        steps.append(('cat', Pipeline([('i', SimpleImputer(strategy='constant', fill_value='NA')),
                                       ('o', OneHotEncoder(handle_unknown='ignore',
                                                           min_frequency=30))]), cat))
    prep = ColumnTransformer(steps)
    clf = (LogisticRegression(max_iter=4000, C=C, class_weight='balanced') if kind == 'logreg'
           else HistGradientBoostingClassifier(max_depth=4, learning_rate=0.06,
                                               max_iter=300, random_state=42))
    return Pipeline([('prep', prep), ('clf', clf)]), num + cat

def gini(y, p):
    return 2 * roc_auc_score(y, p) - 1

def ks(y, p):
    d = pd.DataFrame({'y': np.asarray(y), 'p': np.asarray(p)}).sort_values('p')
    cb = (d.y == 1).cumsum() / max((d.y == 1).sum(), 1)
    cg = (d.y == 0).cumsum() / max((d.y == 0).sum(), 1)
    return float((cg - cb).abs().max())

results = []
models = {}
for kind, label in [('logreg', 'Логистическая регрессия'),
                    ('hgb', 'Градиентный бустинг')]:
    m, cols = build(FEATS, kind)
    m.fit(train[cols], train.target)
    p_tr = m.predict_proba(train[cols])[:, 1]
    p_te = m.predict_proba(test[cols])[:, 1]
    models[kind] = (m, cols, p_te)
    results.append({'модель': label,
                    'Gini train': round(gini(train.target, p_tr), 3),
                    'Gini test': round(gini(test.target, p_te), 3),
                    'разрыв': round(gini(train.target, p_tr) - gini(test.target, p_te), 3),
                    'KS test': round(ks(test.target, p_te), 3)})

display(pd.DataFrame(results))
print('Разрыв между обучением и проверкой показывает переобучение.')
print('Отрицательный разрыв — признак того, что проверочная выборка легче,')
print('либо что схема измерения устроена неверно.')

**Устойчива ли разница между двумя моделями.** Бустинг показал Gini на проверке выше, чем регрессия. Прежде чем выбирать его, нужно понять, отличается ли разница от случайной. Парный бутстрэп пересобирает одну и ту же проверочную выборку тысячу раз и каждый раз считает разницу метрик — если ноль попадает внутрь интервала, разница не доказана.

In [ ]:
# Парный бутстрэп: значима ли разница между логрегом и бустингом на проверке
rng = np.random.default_rng(42)
y_te = test.target.values
p_lr = models['logreg'][2]
p_gb = models['hgb'][2]
n = len(y_te)

obs = gini(y_te, p_gb) - gini(y_te, p_lr)
diffs = []
for _ in range(1000):
    idx = rng.integers(0, n, n)
    yy = y_te[idx]
    if yy.sum() == 0 or yy.sum() == n:      # вырожденная выборка — пропускаем
        continue
    diffs.append(gini(yy, p_gb[idx]) - gini(yy, p_lr[idx]))
diffs = np.array(diffs)
lo, hi = np.percentile(diffs, [2.5, 97.5])

print('Gini на проверке: логрег %.3f | бустинг %.3f' % (gini(y_te, p_lr), gini(y_te, p_gb)))
print('Разница (бустинг − логрег): %+.3f' % obs)
print('95%% интервал парного бутстрэпа: [%+.3f; %+.3f]' % (lo, hi))
print('Доля пересборок, где бустинг впереди: %.1f%%' % (100 * (diffs > 0).mean()))
print()
if lo <= 0 <= hi:
    print('Ноль внутри интервала — разница не доказана.')
    print('Тогда модель выбирается по другим соображениям: логистическая регрессия')
    print('даёт коэффициенты, объяснима бизнесу и проще в сопровождении.')
else:
    print('Ноль вне интервала — разница устойчива на этой выборке.')
    print('Но она измерена на одном проверочном периоде: подтвердить в разделе 17.1,')
    print('где обе модели проверяются на двух периодах сразу.')


### 11.1 На чём держится метрика

На прежней версии таргета (порог 90 дней, без лага) Gini был 0,846 — подозрительно высоко для оттока. Причина была в устройстве таргета: условие «тишина не меньше 90 дней к концу окна» выполнялось автоматически для каждого, кто не совершил ни одной операции внутри окна. Таргет фактически сводился к «нет операций в ближайшие 90 дней», а признак `recency_days` — сколько дней клиент молчит на дату снимка — был почти тавтологичен ему.

С лагом 30 дней этой тавтологии больше нет: между последним известным действием и окном исхода лежит месяц, о котором модель ничего не знает. Метрика упала до значений, типичных для честно поставленной задачи оттока.

Проверка ниже остаётся нужной, но отвечает уже на другой вопрос: **осталась ли давность доминирующим признаком**. Смотрим три вещи: силу отдельных признаков, качество модели без признаков давности и распределение давности по классам. Если и теперь всё держится на `recency_days`, значит лаг помог недостаточно.

In [ ]:
# --- одномерная сила признаков
uni = []
for col in FEATS:
    s = train[col]
    if pd.api.types.is_numeric_dtype(s) and s.notna().sum() > 200 and s.nunique() > 2:
        v = s.fillna(s.median())
        uni.append({'признак': col, 'одномерный_Gini': round(abs(gini(train.target, v)), 3)})
uni = pd.DataFrame(uni).sort_values('одномерный_Gini', ascending=False)
display(uni.head(15))

# --- модель без признаков давности
RECENCY_LIKE = [c for c in FEATS if 'recency' in c or c == 'avg_gap_90']
print('Признаки давности:', RECENCY_LIKE)
FEATS_NR = [c for c in FEATS if c not in RECENCY_LIKE]

rows = []
for label, cols_use in [('все признаки', FEATS), ('без признаков давности', FEATS_NR),
                        ('только recency_days', ['recency_days'])]:
    mm, cc = build(cols_use, 'logreg')
    mm.fit(train[cc], train.target)
    ptr = mm.predict_proba(train[cc])[:, 1]
    pte = mm.predict_proba(test[cc])[:, 1]
    rows.append({'набор': label, 'признаков': len(cols_use),
                 'Gini train': round(gini(train.target, ptr), 3),
                 'Gini test': round(gini(test.target, pte), 3)})
display(pd.DataFrame(rows))
print('Если «только recency_days» даёт почти столько же, сколько полный набор —')
print('модель фиксирует уже начавшееся молчание, а не предсказывает будущее.')
print()

# --- распределение давности по классам
print('Медиана дней молчания на дату снимка:')
display(train.groupby('target').recency_days.describe(
    percentiles=[.25, .5, .75, .9])[['count', '25%', '50%', '75%', '90%', 'max']].round(1))

fig, ax = plt.subplots(figsize=(10, 4.5))
for t, lab in [(0, 'остались'), (1, 'ушли')]:
    ax.hist(train[train.target == t].recency_days.clip(upper=90), bins=45,
            alpha=.55, density=True, label=lab)
ax.set_xlabel('дней молчания на дату снимка'); ax.set_ylabel('плотность')
ax.set_title('Давность последней операции по классам'); ax.legend()
plt.tight_layout(); plt.show()

### 11.2 Что делать с этим результатом

Проверка выше показала, что метрика на давности **не держится**. Три числа:

| Набор | Gini на проверке |
|---|---|
| все 52 признака | 0,624 |
| без четырёх признаков давности | 0,623 |
| только `recency_days` | 0,425 |

Снятие всех признаков давности стоит 0,001 Gini. Одна давность в одиночку даёт 0,425 — две трети от полного набора, но не его. Для сравнения, на прежней версии таргета (без лага) одна давность вытягивала 0,724 из 0,846, то есть модель почти целиком стояла на ней. Лаг эту опору убрал.

Распределения подтверждают: медиана молчания у ушедших 8 дней, у оставшихся 4, а нижний квартиль почти совпадает — 3 против 2. На гистограмме классы перекрываются почти целиком. Ушедшие на дату снимка не выглядят молчунами.

**Что это значит для защиты.** Развилки «оставить как есть или строить вторую модель без давности» больше нет: вторая модель уже фактически построена и даёт тот же результат. Модель отвечает на полезный вопрос — кто уйдёт из тех, кто сейчас активен, — а не фиксирует уже начавшийся уход.

Ниже остаётся отдельный сценарий на подвыборке активных клиентов: он показывает то же самое ещё раз, на срезе, где давность заведомо мала.

In [ ]:
# Отдельный сценарий: только клиенты, активные на дату снимка
ACTIVE_MAX_SILENCE = 30
tr_act = train[train.recency_days <= ACTIVE_MAX_SILENCE]
te_act = test[test.recency_days <= ACTIVE_MAX_SILENCE]

print('Активные клиенты (молчание <= %d дней):' % ACTIVE_MAX_SILENCE)
print('  обучение: %d строк, отток %.2f%%' % (len(tr_act), 100 * tr_act.target.mean()))
print('  проверка: %d строк, отток %.2f%%' % (len(te_act), 100 * te_act.target.mean()))
print()

if tr_act.target.sum() > 30 and te_act.target.sum() > 10:
    rows = []
    for label, cols_use in [('все признаки', FEATS), ('без давности', FEATS_NR)]:
        mm, cc = build(cols_use, 'logreg')
        mm.fit(tr_act[cc], tr_act.target)
        pte = mm.predict_proba(te_act[cc])[:, 1]
        rows.append({'набор': label, 'Gini test (активные)': round(gini(te_act.target, pte), 3)})
    display(pd.DataFrame(rows))
    print('Это честная оценка способности предсказывать уход ЗАРАНЕЕ,')
    print('а не фиксировать уже начавшееся молчание.')
else:
    print('Ушедших среди активных слишком мало для устойчивой оценки.')

## 12. Проверка на пересечение клиентов

90% клиентов проверочного снимка присутствовали и в обучающем. Считаем метрику отдельно на тех, кого модель не видела.

**Оговорка при трактовке.** Новые клиенты — не случайная подвыборка, а другая популяция: они моложе и уходят чаще (9,38% против 5,77%). Более низкая метрика на них не доказывает запоминание людей, она может отражать разницу популяций.

In [ ]:
BEST = 'logreg'
m, cols, p_te = models[BEST]

new_mask = test.is_new_client == 1
old_mask = ~new_mask

rows = [
    {'выборка': 'весь проверочный снимок', 'строк': len(test),
     'отток_%': round(100 * test.target.mean(), 2),
     'Gini': round(gini(test.target, p_te), 3)},
    {'выборка': 'клиенты, бывшие в обучении', 'строк': int(old_mask.sum()),
     'отток_%': round(100 * test.target[old_mask].mean(), 2),
     'Gini': round(gini(test.target[old_mask], p_te[old_mask.values]), 3)},
    {'выборка': 'клиенты, которых модель не видела', 'строк': int(new_mask.sum()),
     'отток_%': round(100 * test.target[new_mask].mean(), 2),
     'Gini': round(gini(test.target[new_mask], p_te[new_mask.values]), 3)},
]
display(pd.DataFrame(rows))

## 12.1 Полный набор метрик

Gini и KS измеряют ранжирование. Для отчёта нужны и пороговые метрики — Precision, Recall, F1, — но у них есть особенность: они зависят от выбранного порога, а порог задаётся экономикой, а не статистикой.

Поэтому считаем их не при пороге 0,5 (он бессмысленен при доле оттока около 5%), а при той глубине обработки, которую даёт экономика: топ-K% списка.

In [ ]:
from sklearn.metrics import (precision_score, recall_score, f1_score,
                             confusion_matrix, precision_recall_curve, average_precision_score)

m_best, cols_best, p_best = models[BEST]
y_true = test.target.values

rows = []
for share in [0.05, 0.07, 0.10, 0.15, 0.20]:
    thr = np.quantile(p_best, 1 - share)
    y_hat = (p_best >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_hat).ravel()
    rows.append({'глубина_%': int(share * 100), 'порог P': round(float(thr), 3),
                 'Precision': round(precision_score(y_true, y_hat, zero_division=0), 3),
                 'Recall': round(recall_score(y_true, y_hat), 3),
                 'F1': round(f1_score(y_true, y_hat), 3),
                 'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn})
display(pd.DataFrame(rows))

print('ROC-AUC : %.3f' % roc_auc_score(y_true, p_best))
print('Gini    : %.3f' % gini(y_true, p_best))
print('KS      : %.3f' % ks(y_true, p_best))
print('PR-AUC  : %.3f (базовый уровень %.3f — доля оттока)'
      % (average_precision_score(y_true, p_best), y_true.mean()))
print()
print('При редком событии PR-AUC информативнее ROC-AUC: она показывает качество')
print('именно на положительном классе, а не усреднённое по обоим.')
print()
print('Порог 0,5 для сравнения:')
y50 = (p_best >= 0.5).astype(int)
print('  Precision %.3f | Recall %.3f | F1 %.3f | предсказано ушедших: %d'
      % (precision_score(y_true, y50, zero_division=0), recall_score(y_true, y50),
         f1_score(y_true, y50), int(y50.sum())))
print('  Порог 0,5 не имеет смысла при доле оттока %.1f%%: он либо ловит почти всех,'
      % (100 * y_true.mean()))
print('  либо почти никого. Рабочий порог задаёт экономика, раздел 15.')

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
pr, rc_, _ = precision_recall_curve(y_true, p_best)
ax[0].plot(rc_, pr); ax[0].axhline(y_true.mean(), ls='--', color='grey', lw=1)
ax[0].set_xlabel('Recall'); ax[0].set_ylabel('Precision'); ax[0].set_title('Precision-Recall')
d = pd.DataFrame(rows)
ax[1].plot(d['глубина_%'], d.Precision, marker='o', label='Precision')
ax[1].plot(d['глубина_%'], d.Recall, marker='o', label='Recall')
ax[1].plot(d['глубина_%'], d.F1, marker='o', label='F1')
ax[1].set_xlabel('глубина обработки, %'); ax[1].legend(); ax[1].set_title('Метрики от глубины')
plt.tight_layout(); plt.show()

## 12.2 Анализ выживаемости

Бинарная классификация отвечает на вопрос «уйдёт ли», выживаемость — на вопрос «когда» и умеет работать с наблюдениями, по которым исход ещё не наступил.

**Цензурирование — главная тонкость этого раздела, и здесь легко ошибиться.** Если считать событием любую последнюю операцию клиента, событие получат все 100% наблюдений: у последней операции по определению нет следующей. Цензурированных окажется ноль, кривые Каплана-Мейера обвалятся у правого края данных, а модель Кокса выдаст бессмысленные коэффициенты — например, что рост числа активных месяцев **повышает** риск ухода.

Правильно: пауза засчитывается как уход, только если после её начала осталось не меньше `N_SILENCE` дней наблюдения. Клиент, чья последняя операция была за две недели до конца данных, не ушёл — мы просто не успели это увидеть, и его наблюдение цензурируется. Отдельно возвращаются в выборку клиенты без единой операции после снимка: наивная версия их теряла, а это ровно те, кто ушёл сразу.

Проверка корректности одна: доля событий должна быть заметно ниже 100%.

In [ ]:
# Время до события: от даты снимка до начала паузы длиной не менее N_SILENCE дней.
#
# ВАЖНО про цензурирование. Наивная версия «событие = первая пауза либо последняя
# операция» засчитывает событие всем без исключения: у последней операции любого
# клиента следующей нет, и условие срабатывает автоматически. Тогда цензурированных
# ноль, а весь смысл анализа выживаемости — именно в них.
#
# Правильная логика: пауза считается уходом, только если после её начала осталось
# не меньше N_SILENCE дней наблюдения. Иначе мы просто не успели увидеть, вернётся
# клиент или нет, и наблюдение цензурируется на горизонте.

HORIZON = (pd.Timestamp(DATA_END) - pd.Timestamp(S_ref)).days
print('Горизонт наблюдения от снимка %s: %d дней' % (S_ref, HORIZON))

surv_raw = q("""
    WITH acts AS (
        SELECT customer_id, txn_ts::date AS d FROM %(txn)s t WHERE 1=1 %(succ)s
    ), after AS (
        SELECT customer_id, d,
               lead(d) OVER (PARTITION BY customer_id ORDER BY d) AS nxt
        FROM acts WHERE d > DATE '%(S)s'
    )
    SELECT customer_id,
           min(CASE WHEN nxt IS NOT NULL AND nxt - d >= %(N)d
                    THEN d END) - DATE '%(S)s'          AS t_gap_inner,
           max(d) - DATE '%(S)s'                        AS t_last
    FROM after GROUP BY 1
""" % dict(txn=TXN, succ=SUCC, S=S_ref, N=N_SILENCE))

# База: все клиенты снимка, включая тех, у кого после снимка операций нет вовсе —
# именно они и есть ушедшие, наивная версия их теряла.
base_s = targets[list(SNAPSHOTS)[1]][['customer_id', 'segment', 'is_payroll',
                                      'acq_channel']].copy()
surv = base_s.merge(surv_raw, on='customer_id', how='left')
surv['t_last'] = surv.t_last.fillna(0)          # нет операций после снимка

# Кандидат на событие: либо пауза внутри данных, либо пауза после последней операции —
# но вторая засчитывается, только если она успела продлиться N_SILENCE дней.
tail_ok = (HORIZON - surv.t_last) >= N_SILENCE
surv['t_event'] = surv.t_gap_inner
surv.loc[surv.t_event.isna() & tail_ok, 't_event'] = surv.t_last

surv['event'] = surv.t_event.notna().astype(int)
surv['duration'] = surv.t_event.fillna(surv.t_last).clip(lower=1)

print('Наблюдений: %d, событий: %d (%.1f%%), цензурировано: %d (%.1f%%)'
      % (len(surv), surv.event.sum(), 100 * surv.event.mean(),
         int((surv.event == 0).sum()), 100 * (1 - surv.event.mean())))
print('Из них без единой операции после снимка: %d' % int((surv.t_last == 0).sum()))
print()
print('Доля событий должна быть заметно ниже 100%. Если она снова 100%,')
print('значит цензурирование не работает и кривые читать нельзя.')

def km(dur, ev):
    """Оценка Каплана-Мейера, векторизованная (без цикла по датам)."""
    df = pd.DataFrame({'t': np.asarray(dur), 'e': np.asarray(ev)})
    g = df.groupby('t').agg(d=('e', 'sum'), n_ev=('e', 'size')).sort_index()
    at_risk = len(df) - g.n_ev.cumsum().shift(fill_value=0)
    surv_p = (1 - g.d / at_risk.replace(0, np.nan)).fillna(1).cumprod()
    return g.index.values, surv_p.values

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for seg, g in surv.groupby('segment'):
    t, s = km(g.duration.values, g.event.values)
    ax[0].step(t, s, where='post', label='%s (n=%d)' % (seg, len(g)))
ax[0].set_title('Выживаемость по сегментам'); ax[0].set_xlabel('дней от снимка')
ax[0].set_ylabel('доля не ушедших'); ax[0].legend(fontsize=8)
ax[0].set_xlim(0, HORIZON - N_SILENCE)

for pr_, g in surv.groupby('is_payroll'):
    t, s = km(g.duration.values, g.event.values)
    ax[1].step(t, s, where='post',
               label='%s (n=%d)' % ('зарплатный' if pr_ else 'не зарплатный', len(g)))
ax[1].set_title('Выживаемость: зарплатный клиент'); ax[1].set_xlabel('дней от снимка')
ax[1].legend(fontsize=8); ax[1].set_xlim(0, HORIZON - N_SILENCE)
plt.tight_layout(); plt.show()

print('Кривые обрезаны на %d днях: дальше уходы неотличимы от края данных.'
      % (HORIZON - N_SILENCE))
print('Расхождение кривых означает, что группы различаются не только долей ушедших,')
print('но и скоростью ухода. Это то, чего бинарная модель не показывает.')


In [ ]:
# Модель пропорциональных рисков Кокса.
# lifelines АВТОМАТИЧЕСКИ НЕ УСТАНАВЛИВАЕТСЯ: сборка под свежий Python может
# занять минуты и выглядеть как зависание. Если библиотеки нет — ячейка
# сообщит об этом и пропустит расчёт; кривые Каплана-Мейера выше остаются в силе.
# Установить вручную:  %pip install lifelines
import importlib
if importlib.util.find_spec('lifelines') is None:
    print('lifelines не установлен — модель Кокса пропущена.')
    print('Чтобы построить её, выполните в отдельной ячейке: %pip install lifelines')
    print('Кривые Каплана-Мейера выше не требуют этой библиотеки.')
else:
    from lifelines import CoxPHFitter
    COX_FEATS = [c for c in ['cnt_30', 'cnt_90', 'trend_30_90', 'ev_30',
                             'active_months_6m', 'mcc_variety_6m', 'tenure_days',
                             'chg_failed_90', 'tickets_90']
                 if c in full.columns]
    cx = (surv[['customer_id', 'duration', 'event']]
          .merge(full[full.snapshot_label == list(SNAPSHOTS)[1]][['customer_id'] + COX_FEATS],
                 on='customer_id', how='inner').dropna())
    if len(cx) > 20000:
        cx = cx.sample(20000, random_state=42)
        print('Выборка ограничена 20 000 наблюдений для скорости.')
    try:
        cph = CoxPHFitter(penalizer=0.1)
        cph.fit(cx.drop(columns=['customer_id']), duration_col='duration', event_col='event')
        display(cph.summary[['coef', 'exp(coef)', 'p']].round(4).sort_values('coef'))
        print()
        print('exp(coef) — во сколько раз меняется мгновенный риск ухода при росте')
        print('признака на единицу. Значение меньше 1 означает защитный эффект.')
        print('Concordance (аналог AUC для выживаемости): %.3f' % cph.concordance_index_)
    except Exception as e:
        print('Модель Кокса не сошлась: %s' % str(e)[:200])

## 12.3 Uplift: оценка эффекта удержания

Uplift — разница между вероятностью остаться при контакте и без него. Задача причинного вывода, а не предсказания: нас интересует не «уйдёт ли клиент», а «изменит ли что-нибудь наш звонок».

### Проблема идентификации и как мы её решаем

Контакты в исторических данных не рандомизированы: банк сам выбирал, кому звонить. Наивная разница между контактированными и остальными измеряет отбор, а не эффект.

**Наш ход — контакт внутри лага.** Схема таргета даёт естественную последовательность:

```
[ признаки ] t0 [ ---- лаг 30 дн. ---- ][ ---- окно исхода ---- ]
   до t0          здесь случился контакт        здесь измеряем
                      (воздействие)                  исход
```

Признаки посчитаны строго **до** контакта, исход — строго **после**. Это устраняет обратную причинность: признаки не могут быть следствием звонка. Остаётся смешение по наблюдаемым характеристикам, и с ним работаем стандартными средствами.

**Допущение, на котором держится оценка:** при равных наблюдаемых признаках назначение контакта не связано с потенциальным исходом. Проверяем его двумя способами — балансом ковариат после взвешивания и анализом чувствительности к скрытому смешению.

In [ ]:
# ---------- Сборка выборки для uplift ----------
# Воздействие: контакт внутри лага, то есть в (t0, t0 + LAG].
# Исход: клиент остался (1) или ушёл (0) в окне исхода.
camp_raw = q("""
    SELECT customer_id, contacted_at::date AS contacted_at,
           campaign_code, channel, responded
    FROM public.churn_retention_campaigns
""")

up_parts = []
for lb, S in SNAPSHOTS.items():
    t0 = pd.Timestamp(S)
    lag_end = t0 + pd.Timedelta(days=int(LAG_DAYS))
    treated = camp_raw[(pd.to_datetime(camp_raw.contacted_at) > t0) &
                       (pd.to_datetime(camp_raw.contacted_at) <= lag_end)]
    treated = treated.groupby('customer_id').agg(
        campaign_code=('campaign_code', 'first'),
        channel=('channel', 'first'),
        responded=('responded', 'max')).reset_index()

    d = full[full.snapshot_label == lb].copy()
    d = d.merge(treated, on='customer_id', how='left')
    d['treated'] = d.campaign_code.notna().astype(int)
    d['retained'] = 1 - d.target          # исход: остался
    up_parts.append(d)

up = pd.concat(up_parts, ignore_index=True)

print('Выборка для uplift: %d наблюдений' % len(up))
print('  контакт внутри лага : %d (%.2f%%)' % (up.treated.sum(), 100 * up.treated.mean()))
print('  остались            : %.2f%%' % (100 * up.retained.mean()))
print()
display(up.groupby('treated').agg(
    наблюдений=('retained', 'size'),
    остались=('retained', 'sum'),
    доля_оставшихся_pct=('retained', lambda s: round(100 * s.mean(), 2))
).rename(index={0: 'контроль', 1: 'контакт'}))

naive_lift = 100 * (up[up.treated == 1].retained.mean() - up[up.treated == 0].retained.mean())
print()
print('Наивная разница: %+.2f п.п. — это ЕЩЁ НЕ uplift, в ней сидит отбор.' % naive_lift)
print('Корректируем ниже.')

if up.treated.sum() < 100:
    print()
    print('[!] Контактов внутри лага мало (%d). Оценка будет шумной.' % up.treated.sum())
    print('    Ниже считается доверительный интервал — смотрите на его ширину.')

### 12.3.0 Проверка идентифицируемости: можно ли вообще оценивать эффект

Прежде чем строить модель склонности, надо убедиться, что задача причинного вывода здесь в принципе решается. Условие идентификации — назначение воздействия зависит только от наблюдаемых признаков, но не от самого исхода.

Первый прогон дал повод усомниться: среди контактированных остались 2,45%, в контроле — 94,38%. Разрыв в 92 процентных пункта невозможно объяснить отбором по признакам активности. Так выглядит не смещение отбора, а **назначение по исходу**: банк контактирует тех, кто уже ушёл.

Подозрение подтверждается названиями кампаний: `WINBACK_CASHBACK` и `WINBACK_10` — это возврат ушедших, а не удержание. Оценивать по ним эффект удержания — то же самое, что оценивать эффективность реанимации по смертности в реанимации.

Ячейка ниже разделяет кампании на удерживающие и возвратные и смотрит на долю оставшихся отдельно по каждой группе, а заодно проверяет, в какой момент относительно последней операции происходит контакт.

**Как читать результат.** Если у удерживающих кампаний доля оставшихся сопоставима с контролем — uplift можно строить на этой подвыборке, дальнейшие разделы работают. Если она тоже близка к нулю — эффект на этих данных не идентифицируется, и разделы 12.3.1–12.3.7 надо не чинить, а заменить честным разбором того, почему это так, с переходом к дизайну эксперимента в 12.3.8.

In [ ]:
# ---------- Разделяем кампании на удерживающие и возвратные ----------
RETAIN_CODES = ['FEE_WAIVER_3M', 'CALL_RETENTION']   # удержание активных
WINBACK_CODES = ['WINBACK_CASHBACK', 'WINBACK_10']   # возврат ушедших

tr = up[up.treated == 1].copy()
tr['тип_кампании'] = np.where(tr.campaign_code.isin(WINBACK_CODES), 'возврат',
                     np.where(tr.campaign_code.isin(RETAIN_CODES), 'удержание', 'прочее'))

by_type = tr.groupby('тип_кампании').agg(
    контактов=('retained', 'size'),
    остались=('retained', 'sum'),
    доля_оставшихся_pct=('retained', lambda s: round(100 * s.mean(), 2))
)
ctrl = up[up.treated == 0]
by_type.loc['КОНТРОЛЬ (без контакта)'] = [len(ctrl), int(ctrl.retained.sum()),
                                          round(100 * ctrl.retained.mean(), 2)]
display(by_type)

print()
print('Разбивка по конкретным кампаниям:')
display(tr.groupby('campaign_code').agg(
    контактов=('retained', 'size'),
    доля_оставшихся_pct=('retained', lambda s: round(100 * s.mean(), 2))
).sort_values('доля_оставшихся_pct'))

# ---------- В какой момент происходит контакт ----------
# Если контакт систематически приходится на клиентов, уже давно молчащих,
# это прямое свидетельство назначения по исходу.
if 'recency_days' in up.columns:
    print()
    print('Давность последней операции на дату снимка (медиана):')
    cmp_rec = pd.DataFrame({
        'медиана recency_days': [
            round(ctrl.recency_days.median(), 1),
            round(tr[tr.тип_кампании == 'удержание'].recency_days.median(), 1)
            if (tr.тип_кампании == 'удержание').any() else np.nan,
            round(tr[tr.тип_кампании == 'возврат'].recency_days.median(), 1)
            if (tr.тип_кампании == 'возврат').any() else np.nan],
    }, index=['контроль', 'контакт: удержание', 'контакт: возврат'])
    display(cmp_rec)

# ---------- Вердикт ----------
ret_mask = tr.тип_кампании == 'удержание'
print()
if ret_mask.sum() < 30:
    print('[!] Удерживающих контактов внутри лага меньше 30 (%d).' % int(ret_mask.sum()))
    print('    Подвыборки не хватит даже на грубую оценку.')
else:
    r_ret = 100 * tr.loc[ret_mask, 'retained'].mean()
    r_ctl = 100 * ctrl.retained.mean()
    print('Удерживающие кампании: осталось %.2f%% против %.2f%% в контроле (%d контактов).'
          % (r_ret, r_ctl, int(ret_mask.sum())))
    print()
    if r_ret < 0.5 * r_ctl:
        print('ВЕРДИКТ: разрыв слишком велик и для удерживающих кампаний.')
        print('Назначение контакта зависит от исхода, а не только от признаков.')
        print('Условие идентификации нарушено — разделы 12.3.1-12.3.7 неприменимы,')
        print('корректный ответ даёт 12.3.8: дизайн эксперимента.')
    else:
        print('ВЕРДИКТ: на удерживающих кампаниях разрыв объясним отбором.')
        print('Uplift можно строить на этой подвыборке — задайте UPLIFT_SUBSET ниже.')


### 12.3.1 Модель склонности: формальная проверка условия позитивности

Раздел 12.3.0 показал проблему по долям. Здесь она измеряется формально.

Причинный вывод из наблюдательных данных требует двух условий. Первое — **сопоставимость**: назначение воздействия зависит только от наблюдаемых признаков. Второе — **позитивность**: у любого клиента есть ненулевая вероятность и получить контакт, и не получить его. Если для контактированных не существует похожих неконтактированных, сравнивать не с чем, и никакое взвешивание этого не исправит.

Модель склонности проверяет оба условия сразу. Её AUC показывает, насколько предсказуемо назначение контакта, а перекрытие распределений склонности — есть ли вообще область, где группы сопоставимы.

**Как читать.** AUC около 0,5 — назначение почти случайное. AUC выше 0,8 — отбор настолько сильный, что контактированные и контрольные почти не пересекаются, и оценка эффекта будет держаться на экстраполяции модели, а не на данных.

In [ ]:
UP_FEATS = [c for c in FEATS if c in up.columns]

ps_model, ps_cols = build(UP_FEATS, 'logreg')
ps_model.named_steps['clf'].set_params(class_weight=None)
ps_model.fit(up[ps_cols], up.treated)
up['ps'] = ps_model.predict_proba(up[ps_cols])[:, 1]

auc_ps = roc_auc_score(up.treated, up.ps)
print('Модель склонности: AUC %.3f (Gini %.3f)' % (auc_ps, 2 * auc_ps - 1))
if auc_ps < 0.60:
    print('  Назначение контакта слабо предсказуемо — отбор умеренный,')
    print('  условие сопоставимости правдоподобно.')
else:
    print('  Отбор выраженный: банк целенаправленно выбирал, кому звонить.')
    print('  Коррекция обязательна, а вывод остаётся условным.')
print()

# --- общая опора
lo = max(up[up.treated == 1].ps.min(), up[up.treated == 0].ps.min())
hi = min(up[up.treated == 1].ps.max(), up[up.treated == 0].ps.max())
up['in_support'] = up.ps.between(lo, hi)
print('Область общей опоры: склонность от %.4f до %.4f' % (lo, hi))
print('Внутри опоры: %d из %d наблюдений (%.1f%%), контактов %d'
      % (up.in_support.sum(), len(up), 100 * up.in_support.mean(),
         int(up[up.in_support].treated.sum())))

fig, ax = plt.subplots(figsize=(10, 4))
for t, lab in [(0, 'контроль'), (1, 'контакт')]:
    ax.hist(up[up.treated == t].ps, bins=40, alpha=.55, density=True, label=lab)
ax.axvline(lo, ls='--', color='grey', lw=1); ax.axvline(hi, ls='--', color='grey', lw=1)
ax.set_xlabel('оценка склонности к контакту'); ax.set_ylabel('плотность')
ax.set_title('Перекрытие групп по склонности'); ax.legend()
plt.tight_layout(); plt.show()
print('Чем сильнее перекрываются распределения, тем надёжнее сравнение.')

### 12.3.2 Почему коррекция здесь не спасает и что делать вместо неё

Изначально в этом месте стояли шесть подразделов: взвешивание с проверкой баланса, средний эффект через AIPW, индивидуальный uplift через T-learner, Qini-кривая, сегментация на четыре квадранта и E-value. Все они предполагают, что условия из 12.3.1 выполнены хотя бы приблизительно. Диагностика показала, что не выполнены — поэтому запускать их означало бы получить число, которое выглядит как результат, но им не является.

**Что именно сломано.** Разбивка из 12.3.0:

| Группа | Контактов | Осталось |
|---|---|---|
| Возврат (`WINBACK_CASHBACK`, `WINBACK_10`) | 103 | **0,00%** |
| Удержание (`CALL_RETENTION`, `FEE_WAIVER_3M`) | 101 | 4,95% |
| Контроль (без контакта) | 30 941 | 94,38% |

Ноль из 103 у возвратных кампаний — не маленький эффект, а определение: возврат применяют к тем, кто уже ушёл. Но и удерживающие кампании дают 4,95% против 94,38%.

**Прямое доказательство назначения по исходу** — медианная давность последней операции на дату снимка: 4 дня у контроля против 34 дней у контактированных, и это одинаково для удерживающих и возвратных кампаний. Банк связывается с теми, кто уже месяц молчит. При пороге тишины 60 дней такой клиент почти гарантированно попадёт в отток независимо от того, позвонили ему или нет.

**Числа прогона.** Модель склонности отличает контактированных от остальных с AUC 0,940. Это почти детерминированное назначение: банк не «склонен» звонить одним чаще, чем другим, — он звонит по правилу.

Формальная проверка общей опоры при этом проходит: диапазон склонности у контроля от 0,0004 до 0,802 накрывает все 204 контакта. **Это ловушка.** Проверка по минимуму и максимуму сравнивает границы диапазонов, а не плотности. На гистограмме видно, что контроль сосредоточен у нуля, а контакты размазаны по всей шкале до 0,8 — формально пересечение есть, фактически сравнивать не с кем.

Разоблачает это эффективный размер выборки: у контактированных n = 204, но ESS = 23,6, то есть 11,6% от числа наблюдений, и одно наблюдение забирает 12,6% суммарного веса. У контроля ESS 99,8% от n. Иначе говоря, взвешенная оценка эффекта опиралась бы примерно на два десятка человек, из которых один весит как восьмая часть всей группы.

**Почему это не лечится взвешиванием.** Обратное взвешивание по склонности переносит вес с контрольных наблюдений на те, что похожи на контактированных. Оно работает, когда похожие есть. Здесь контакт получают клиенты из хвоста распределения давности, где контрольных почти нет, поэтому веса концентрируются на единицах наблюдений. Ячейка ниже показывает это числом — через эффективный размер выборки.

**Что это значит для ответа ментору.** Требование звучало как «uplift, сценарий взаимодействуем / не взаимодействуем». Честный ответ: **на этих данных сценарий «не взаимодействуем» не наблюдаем для тех, с кем взаимодействовали.** Разница между группами измеряет не эффект кампании, а критерий, по которому банк выбирает, кому звонить.

Это не тупик, а результат. Он говорит бизнесу конкретную вещь: программа удержания запускается слишком поздно — через месяц молчания, когда лаг для вмешательства уже израсходован. Что с этим делать, считается в 12.3.8.

In [ ]:
# ---------- Насколько бесполезно взвешивание: эффективный размер выборки ----------
# ESS (effective sample size) показывает, скольким "равноправным" наблюдениям
# эквивалентна взвешенная выборка. Если веса концентрируются на единицах
# наблюдений, ESS обваливается, и любая оценка держится на них одних.

sup = up[up.in_support].copy()
print('В области общей опоры: %d наблюдений, из них контактов %d'
      % (len(sup), int(sup.treated.sum())))

if sup.treated.sum() < 20:
    print()
    print('Контактов внутри общей опоры меньше 20 — взвешивать нечего.')
    print('Это и есть нарушение позитивности в чистом виде.')
else:
    eps = 1e-6
    sup['w'] = np.where(sup.treated == 1,
                        1 / np.clip(sup.ps, eps, 1),
                        1 / np.clip(1 - sup.ps, eps, 1))
    for t, lab in [(1, 'контакт'), (0, 'контроль')]:
        w = sup.loc[sup.treated == t, 'w'].values
        ess = w.sum() ** 2 / (w ** 2).sum()
        print()
        print('%s: n = %d, ESS = %.1f (%.1f%% от n)' % (lab, len(w), ess, 100 * ess / len(w)))
        print('   максимальный вес забирает %.1f%% суммарного веса' % (100 * w.max() / w.sum()))

    print()
    print('Ориентир: ESS ниже половины от n означает, что оценка опирается')
    print('на малую часть наблюдений и её доверительный интервал будет широким.')
    print('ESS в единицах наблюдений означает, что оценки нет вовсе.')

# ---------- Наблюдаемая разница, названная своим именем ----------
r1 = up[up.treated == 1].retained.mean()
r0 = up[up.treated == 0].retained.mean()
print()
print('Наблюдаемая разница: %+.2f п.п.' % (100 * (r1 - r0)))
print('Это НЕ эффект кампании. Это разница между теми, кого банк счёл уходящими,')
print('и всеми остальными. Она измеряет качество отбора банка, а не пользу контакта.')
print()
print('Для сравнения — насколько поздно приходит контакт:')
if 'recency_days' in up.columns:
    print('  медиана дней молчания: контроль %.0f, контакт %.0f'
          % (up[up.treated == 0].recency_days.median(),
             up[up.treated == 1].recency_days.median()))
print('  лаг на вмешательство по конструкции таргета: %d дней' % LAG_DAYS)


### 12.3.8 Что предложить бизнесу

Из 12.3.2 следует два вывода: один про измерение, второй про саму программу удержания.

**Вывод про программу.** Контакт происходит в среднем через 34 дня молчания. При пороге тишины 60 дней это означает, что на вмешательство остаётся меньше месяца, а клиент к этому моменту уже прошёл большую часть пути к уходу. Программа работает как возврат, а называется удержанием. Первая рекомендация не про модель: **сдвинуть момент контакта к дате скоринга**, то есть в первые дни после снимка, а не через месяц молчания. Модель из раздела 11 как раз и даёт список тех, с кем надо связаться заранее.

**Вывод про измерение.** Пока контакт назначается по признаку, который сам является предвестником исхода, эффект программы измерить нельзя — ни взвешиванием, ни AIPW, ни двойной устойчивостью. Нужен эксперимент.

**Дизайн.** Модель ранжирует клиентов по риску. Из отобранной части случайным жребием откладывается контрольная группа — с ней не связываются. Через длительность окна исхода сравниваются доли оставшихся. Разница даёт эффект без допущений о сопоставимости.

**Что зафиксировать до запуска:**
- размер контрольной группы (расчёт ниже);
- момент контакта — внутри лага, а не после;
- длительность и критерий успеха;
- обязательно — **флаг вмешательства в витрине данных**. Без него следующая модель обучится на данных, испорченных этой же кампанией, и повторится ровно та ситуация, которую мы сейчас разбираем.

Расчёт необходимого размера выборки — ниже. Он считается от базовой доли оставшихся среди тех, кто в момент скоринга ещё активен, а не среди контактированных: именно на этой популяции будет работать программа после переноса момента контакта.

In [ ]:
from math import ceil, sqrt

def sample_size(p0, lift_pp, alpha=0.05, power=0.8):
    p1 = min(max(p0 + lift_pp / 100, 1e-6), 1 - 1e-6)
    pbar = (p0 + p1) / 2
    z_a, z_b = 1.96, 0.84
    return ceil((z_a * sqrt(2 * pbar * (1 - pbar)) +
                 z_b * sqrt(p0 * (1 - p0) + p1 * (1 - p1))) ** 2 / (p1 - p0) ** 2)

# Базовая доля: клиенты БЕЗ контакта — на них программа и будет работать
p0 = float(up[up.treated == 0].retained.mean())
print('Базовая доля оставшихся без контакта: %.2f%%' % (100 * p0))
print()
print('Требуемый размер КАЖДОЙ группы (мощность 80%, уровень 5%):')
for lift in [1, 2, 3, 5]:
    n = sample_size(p0, lift)
    print('  эффект %d п.п. -> %s клиентов в группе (всего %s)'
          % (lift, format(n, ',d').replace(',', ' '),
             format(2 * n, ',d').replace(',', ' ')))
print()
print('Сопоставление с масштабом: в топ-5%% риска проверочного снимка %d клиентов.'
      % int(len(test) * 0.05))
print('Исторически банк делал около %d контактов за два года.' % len(camp_raw))
print()
print('Оценённого эффекта мы намеренно НЕ подставляем: из наблюдательных данных')
print('он не идентифицируется (12.3.2). Размер выборки считается от эффекта,')
print('который бизнес считает практически значимым, а не от измеренного.')


## 12.4 Демография и зарплатный признак: сигнал или шум

Проверяем прямо: несут ли анкетные признаки информацию об оттоке сверх поведенческих. Метод — сравнение модели на поведении с моделью на поведении плюс демография.

In [ ]:
DEMO = [c for c in ['gender', 'region', 'segment', 'acq_channel', 'is_payroll',
                    'declared_income_uzs', 'age_years'] if c in FEATS]
BEHAV = [c for c in FEATS if c not in DEMO]
print('Демографических признаков: %d -> %s' % (len(DEMO), DEMO))
print()

rows = []
for label, cols_use in [('только поведение', BEHAV),
                        ('только демография', DEMO),
                        ('поведение + демография', FEATS)]:
    if not cols_use:
        continue
    mm, cc = build(cols_use, 'logreg')
    mm.fit(train[cc], train.target)
    rows.append({'набор': label, 'признаков': len(cols_use),
                 'Gini test': round(gini(test.target,
                                         mm.predict_proba(test[cc])[:, 1]), 3)})
display(pd.DataFrame(rows))
print()

# одномерная сила каждого демографического признака
uni_demo = []
for col in DEMO:
    if pd.api.types.is_numeric_dtype(train[col]):
        v = train[col].fillna(train[col].median())
        g = abs(gini(train.target, v))
    else:
        rate = train.groupby(col).target.mean()
        v = train[col].map(rate).fillna(train.target.mean())
        g = abs(gini(train.target, v))
    uni_demo.append({'признак': col, 'одномерный_Gini': round(g, 3)})
display(pd.DataFrame(uni_demo).sort_values('одномерный_Gini', ascending=False))
print()

# зарплатный клиент отдельно
if 'is_payroll' in train.columns:
    pay = train.groupby('is_payroll').target.agg(['size', 'sum', 'mean'])
    pay.columns = ['клиентов', 'ушли', 'доля']
    pay['доля_pct'] = (100 * pay.доля).round(2)
    pay.index = ['не зарплатный', 'зарплатный']
    display(pay[['клиентов', 'ушли', 'доля_pct']])
    from math import sqrt
    n1, x1 = int(pay.клиентов.iloc[0]), int(pay.ушли.iloc[0])
    n2, x2 = int(pay.клиентов.iloc[1]), int(pay.ушли.iloc[1])
    p1, p2 = x1/n1, x2/n2
    pp = (x1+x2)/(n1+n2)
    z = (p1 - p2) / sqrt(pp*(1-pp)*(1/n1+1/n2))
    print('z = %.2f — %s' % (z, 'различие значимо' if abs(z) > 1.96 else 'различие незначимо, признак шумовой'))
print()
print('Если добавление демографии не меняет Gini, а одномерные значения')
print('близки к нулю — эти признаки шум, и держать их в модели не нужно:')
print('они усложняют скоркарту и создают риск дискриминации.')

**Вывод по 12.4.** Демография значима сама по себе и избыточна в присутствии поведения.

| Набор | Признаков | Gini на проверке |
|---|---|---|
| только поведение | 45 | 0,618 |
| только демография | 7 | 0,284 |
| поведение + демография | 52 | 0,624 |

Семь анкетных признаков в одиночку дают 0,284 — это не ноль. Сильнее всех `is_payroll` (одномерный Gini 0,178): зарплатные клиенты уходят втрое реже, 3,23% против 9,03%, z = 10,41. Различие настоящее.

Но добавление всей демографии к поведенческим признакам даёт **+0,006 Gini**. Это меньше, чем разница, которая уже оказалась невоспроизводимой при выборе модели (0,027). Информация о том, что зарплатный клиент лояльнее, уже содержится в его поведении: он чаще проводит операции, реже уходит в паузы.

`gender` (0,001), `declared_income_uzs` (0,003) и `age_years` (0,017) не несут почти ничего даже поодиночке.

**Практический вывод.** Демографию из модели можно убрать без потери качества. Для банковской модели это не только упрощение: `gender`, `region` и `age_years` среди предикторов — регуляторный риск, и держать их ради 0,006 Gini не стоит. В текущем ноутбуке модель оставлена на полном наборе, чтобы не пересчитывать экономику; в продуктовой версии правильный набор — 45 поведенческих признаков.

**Что не следует говорить:** «зарплатный признак — шум». Он не шум, он дублирующий. Как сегментирующий признак для выбора кампании он остаётся полезным, просто не добавляет модели предсказательной силы.

## 13. Что влияет на отток

In [ ]:
if BEST == 'logreg':
    prep = m.named_steps['prep']
    names = []
    for name, trans, cs in prep.transformers_:
        if name == 'num':
            names += list(cs)
        elif name == 'cat':
            names += list(trans.named_steps['o'].get_feature_names_out(cs))
    imp = (pd.DataFrame({'признак': names, 'коэффициент': m.named_steps['clf'].coef_[0]})
           .assign(модуль=lambda d: d.коэффициент.abs())
           .sort_values('модуль', ascending=False).head(20))
    display(imp[['признак', 'коэффициент']].round(3))
    print('Положительный коэффициент — признак повышает вероятность ухода.')

# одномерная сила признаков — независимо от модели
uni_rows = []
for col in FEATS:
    s = train[col]
    if pd.api.types.is_numeric_dtype(s) and s.notna().sum() > 200 and s.nunique() > 2:
        v = s.fillna(s.median())
        uni_rows.append({'признак': col, 'одномерный_Gini': round(abs(gini(train.target, v)), 3)})
display(pd.DataFrame(uni_rows).sort_values('одномерный_Gini', ascending=False).head(15))

## 14. Кого модель ставит в топ риска

Бизнесу нужен не вердикт, а приоритезированный список. Смотрим, насколько отток концентрируется в верхних децилях.

In [ ]:
dec = test.assign(p=p_te).copy()
dec['дециль'] = pd.qcut(dec.p, 10, labels=False, duplicates='drop')
dec['дециль'] = 10 - dec['дециль']  # 1 = самый рискованный

tbl = dec.groupby('дециль').agg(
    клиентов=('target', 'size'),
    ушли=('target', 'sum'),
    отток_pct=('target', lambda s: round(100 * s.mean(), 2)),
    средняя_P=('p', lambda s: round(s.mean(), 3))).reset_index()
tbl['доля_всех_ушедших_%'] = (100 * tbl.ушли / tbl.ушли.sum()).round(1)
tbl['накопленно_%'] = tbl['доля_всех_ушедших_%'].cumsum().round(1)
display(tbl)

base_rate = test.target.mean()
top1 = tbl.iloc[0]
print('В верхнем дециле отток %.2f%% против среднего %.2f%% — концентрация в %.1f раза.'
      % (top1.отток_pct, 100 * base_rate, top1.отток_pct / (100 * base_rate)))
print('Верхние 20%% клиентов содержат %.1f%% всех ушедших.'
      % tbl['накопленно_%'].iloc[1])

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.bar(tbl.дециль, tbl.отток_pct)
ax.axhline(100 * base_rate, ls='--', color='grey', lw=1)
ax.set_xlabel('дециль риска (1 — самый рискованный)'); ax.set_ylabel('отток, %')
ax.set_title('Концентрация оттока по децилям')
plt.tight_layout(); plt.show()

## 15. Экономика удержания

Бюджета хватает не на всех, поэтому вопрос не «кто уйдёт», а «на кого потратить». Три величины задают ответ: ценность клиента, стоимость контакта и доля откликнувшихся.

Долю отклика берём фактическую — из истории кампаний удержания, а не из предположения.

> **Важная оговорка, которую надо назвать самому.** `RESPONSE_RATE = 12,6%` — это доля отклика на кампанию, а не доказанный эффект удержания. Вся выгода ниже держится на неявном равенстве «откликнулся = остался», которое на этих данных не проверяется: раздел 12.3 показал, что причинный эффект контакта не идентифицируется. Поэтому расчёт ниже — **оценка потенциала при допущении, что отклик означает удержание**, а не измеренная выгода. Величина допущения проверяется в анализе чувствительности, и первым же экспериментом из 12.3.8 его надо заменить на измеренное число.

In [ ]:
camp = q("""
    SELECT campaign_code, channel, count(*) AS контактов,
           sum(responded::int) AS откликнулись
    FROM public.churn_retention_campaigns
    GROUP BY 1, 2 ORDER BY 3 DESC
""")
camp['отклик_%'] = (100 * camp.откликнулись / camp.контактов).round(1)
display(camp)
display(q("SELECT min(contacted_at)::date AS с, max(contacted_at)::date AS по, "
          "count(*) AS всего FROM public.churn_retention_campaigns"))

RESPONSE_RATE = camp.откликнулись.sum() / camp.контактов.sum()
print('Средняя доля отклика: %.1f%%' % (100 * RESPONSE_RATE))

In [ ]:
# --- параметры экономики
# Ценность клиента: месячная маржа × ожидаемый срок жизни.
# Оцениваем по фактическому обороту, а не по предположению.
mon_turnover = test.sum_90.fillna(0) / 3
MARGIN_RATE = 0.02          # доля банка от оборота, ДОПУЩЕНИЕ
HORIZON_MONTHS = 12         # горизонт учёта ценности
CONTACT_COST = 15000        # стоимость одного контакта, сум, ДОПУЩЕНИЕ

test_e = test.assign(p=p_te, clv=mon_turnover * MARGIN_RATE * HORIZON_MONTHS)
print('Медианная ценность клиента за %d мес.: %s сум'
      % (HORIZON_MONTHS, format(int(test_e.clv.median()), ',d').replace(',', ' ')))
print('Стоимость контакта: %s сум | доля отклика: %.1f%%'
      % (format(CONTACT_COST, ',d').replace(',', ' '), 100 * RESPONSE_RATE))
print()

# --- сколько клиентов обрабатывать
rows = []
order = test_e.sort_values('p', ascending=False)
for share in [0.05, 0.10, 0.15, 0.20, 0.30, 0.50, 1.00]:
    k = int(len(order) * share)
    sel = order.head(k)
    saved = sel.target.sum() * RESPONSE_RATE          # удержанных клиентов
    benefit = (sel[sel.target == 1].clv.sum() * RESPONSE_RATE)
    cost = k * CONTACT_COST
    rows.append({'обрабатываем_%': int(share * 100), 'клиентов': k,
                 'ушедших_в_выборке': int(sel.target.sum()),
                 'удержим_ожидаемо': round(saved, 1),
                 'затраты': int(cost), 'выгода': int(benefit),
                 'чистый_эффект': int(benefit - cost)})
econ = pd.DataFrame(rows)
display(econ)

best = econ.loc[econ.чистый_эффект.idxmax()]
print('Оптимум: обрабатывать %d%% списка — чистый эффект %s сум'
      % (best['обрабатываем_%'], format(int(best.чистый_эффект), ',d').replace(',', ' ')))
print()
print('Все три параметра — MARGIN_RATE, HORIZON_MONTHS, CONTACT_COST — допущения.')
print('Их обязан подтвердить продуктовый и финансовый блок. Проверка чувствительности ниже.')

### 15.1 Уточнение оптимума и способа сортировки

Оптимум пришёлся на 5% — наименьшее значение сетки, значит настоящий может лежать левее. Продлеваем сетку вниз.

Одновременно проверяем способ упорядочивания списка. Интуитивно кажется, что сортировать надо по ожидаемым потерям `вероятность × ценность`: так мы приоритезируем не просто уходящих, а дорого уходящих. На практике это работает хуже, и причина в том, что ценность и риск связаны **отрицательно**: чем активнее клиент, тем больше оборот и тем меньше вероятность ухода. Произведение затаскивает в список богатых и лояльных.

Проверяем три схемы: по вероятности, по ожидаемым потерям и гибридную — сначала отбор по вероятности, затем сортировка по ценности внутри отобранных.

In [ ]:
# --- уточнённая сетка
rows = []
order_p = test_e.sort_values('p', ascending=False)
for share in [0.01, 0.02, 0.03, 0.04, 0.05, 0.07, 0.10, 0.15, 0.20]:
    k = int(len(order_p) * share)
    sel = order_p.head(k)
    benefit = sel[sel.target == 1].clv.sum() * RESPONSE_RATE
    cost = k * CONTACT_COST
    rows.append({'обрабатываем_%': round(share * 100, 1), 'клиентов': k,
                 'ушедших': int(sel.target.sum()),
                 'доля_ушедших_в_списке_%': round(100 * sel.target.mean(), 1),
                 'затраты': int(cost), 'выгода': int(benefit),
                 'чистый_эффект': int(benefit - cost)})
fine = pd.DataFrame(rows)
display(fine)

best = fine.loc[fine.чистый_эффект.idxmax()]
print('Оптимум: %.1f%% списка (%d клиентов), чистый эффект %s сум'
      % (best['обрабатываем_%'], best.клиентов,
         format(int(best.чистый_эффект), ',d').replace(',', ' ')))

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(fine['обрабатываем_%'], fine.чистый_эффект / 1e6, marker='o')
ax.axhline(0, ls='--', color='grey', lw=1)
ax.set_xlabel('доля списка, %'); ax.set_ylabel('чистый эффект, млн сум')
ax.set_title('Экономика удержания при разной глубине обработки')
plt.tight_layout(); plt.show()

### 15.2 Сравнение схем сортировки

In [ ]:
K = int(len(test_e) * 0.05)   # глубина обработки из оптимума

# схема 1: по вероятности ухода
s1 = test_e.sort_values('p', ascending=False).head(K)
# схема 2: по ожидаемым потерям
s2 = test_e.assign(loss=test_e.p * test_e.clv).sort_values('loss', ascending=False).head(K)
# схема 3: гибрид — отбор по вероятности вдвое шире, затем топ по ценности
s3 = (test_e.sort_values('p', ascending=False).head(K * 2)
      .sort_values('clv', ascending=False).head(K))

rows = []
for label, s in [('по вероятности ухода', s1),
                 ('по ожидаемым потерям', s2),
                 ('гибрид: риск, затем ценность', s3)]:
    ben = s[s.target == 1].clv.sum() * RESPONSE_RATE
    rows.append({'схема': label,
                 'ушедших_в_списке': int(s.target.sum()),
                 'точность_списка_%': round(100 * s.target.mean(), 1),
                 'выгода': int(ben),
                 'чистый_эффект': int(ben - K * CONTACT_COST)})
display(pd.DataFrame(rows))
print()
print('Ценность и риск связаны отрицательно: активные клиенты тратят больше')
print('и уходят реже. Поэтому произведение «вероятность × ценность» разбавляет')
print('список лояльными клиентами с высоким оборотом. Сортировка по вероятности')
print('с последующим учётом ценности внутри отобранных работает лучше.')

Схема сортировки и глубина обработки связаны: оптимальная доля списка зависит от того, как список упорядочен. Искать их надо совместно, а не по очереди.

In [ ]:
def make_list(scheme, k):
    if scheme == 'вероятность':
        return test_e.sort_values('p', ascending=False).head(k)
    if scheme == 'ожидаемые потери':
        return test_e.assign(l=test_e.p * test_e.clv).sort_values('l', ascending=False).head(k)
    return (test_e.sort_values('p', ascending=False).head(min(k * 2, len(test_e)))
            .sort_values('clv', ascending=False).head(k))

rows = []
for scheme in ['вероятность', 'ожидаемые потери', 'гибрид']:
    for share in [0.01, 0.02, 0.03, 0.05, 0.07, 0.10, 0.15, 0.20]:
        k = int(len(test_e) * share)
        s = make_list(scheme, k)
        ben = s[s.target == 1].clv.sum() * RESPONSE_RATE
        rows.append({'схема': scheme, 'доля_%': round(share * 100, 1),
                     'ушедших': int(s.target.sum()),
                     'чистый_эффект_млн': round((ben - k * CONTACT_COST) / 1e6, 2)})
grid = pd.DataFrame(rows)
display(grid.pivot(index='доля_%', columns='схема', values='чистый_эффект_млн'))

top = grid.loc[grid.чистый_эффект_млн.idxmax()]
print('Максимум: схема «%s», глубина %.1f%%, чистый эффект %.2f млн сум, ушедших в списке %d'
      % (top.схема, top['доля_%'], top.чистый_эффект_млн, top.ушедших))
print()
print('ВАЖНАЯ ОГОВОРКА. Выгода считается как полная годовая ценность удержанного')
print('клиента, а доля отклика принята одинаковой для всех. Второе допущение')
print('сомнительно: удержать клиента с высоким оборотом обычно труднее и дороже.')
print('Если отклик у дорогих клиентов ниже, преимущество схем, учитывающих')
print('ценность, сократится. Проверка ниже.')

# насколько вывод устойчив, если дорогие клиенты откликаются хуже
print()
for penalty in [1.0, 0.75, 0.5]:
    res = []
    for scheme in ['вероятность', 'ожидаемые потери', 'гибрид']:
        best = -1e18
        for share in [0.01, 0.02, 0.03, 0.05, 0.07, 0.10, 0.15]:
            k = int(len(test_e) * share); s = make_list(scheme, k)
            # отклик падает для верхней половины по ценности
            med = test_e.clv.median()
            rr = np.where(s.clv > med, RESPONSE_RATE * penalty, RESPONSE_RATE)
            ben = float((s.clv * (s.target == 1) * rr).sum())
            best = max(best, ben - k * CONTACT_COST)
        res.append('%s %.1f млн' % (scheme, best / 1e6))
    print('отклик дорогих × %.2f:  %s' % (penalty, ' | '.join(res)))

### 15.3 Что делать с каждой группой

Одинаковое предложение всем неэффективно: причины ухода разные, и цена удержания тоже.

| Группа | Доля оттока | Действие | Стоимость |
|---|---|---|---|
| Технический отток | 20,3% | напоминание об обновлении карты, повтор списания | минимальная |
| Затухание активности | 28,2% | продуктовый стимул, кешбэк на привычные категории | средняя |
| Недовольство сервисом | 17,1% | сначала закрыть обращение, потом предлагать | средняя |
| Прочие | 2,0% | не трогать | — |

**Технический отток — первый приоритет.** Клиент не решал уходить: у него не прошло списание, истекла или заблокирована карта. Из 7 154 неудачных списаний в базе 3 570 из-за недостатка средств, 1 824 из-за истёкшей карты, 1 760 из-за блокировки. Последние две категории лечатся уведомлением, то есть почти бесплатно, и это самая выгодная группа при любых допущениях об экономике.

**Прочие трогать не нужно.** Их отток 2% — ниже среднего по базе. Контакт с ними это чистые затраты.

In [ ]:
# Классификация по вероятной причине ухода.
# Порядок проверок задаёт приоритет: технический отток лечится дешевле всего,
# поэтому проверяется первым.
def group(r):
    if r.get('chg_failed_90', 0) > 0:
        return 'технический отток'
    if r.get('tickets_unresolved_180', 0) > 0:
        return 'недовольство сервисом'
    if pd.notna(r.get('trend_30_90')) and r.get('trend_30_90', 1) < 0.6:
        return 'затухание активности'
    return 'прочие'

sel = test_e.sort_values('p', ascending=False).head(K).copy()
sel['группа'] = sel.apply(group, axis=1)

prio = sel.groupby('группа').agg(
    клиентов=('target', 'size'),
    фактический_отток_pct=('target', lambda s: round(100 * s.mean(), 1)),
    средняя_ценность=('clv', lambda s: int(s.mean())),
    ожидаемо_удержим=('target', lambda s: round(s.sum() * RESPONSE_RATE, 1))
).sort_values('фактический_отток_pct', ascending=False)
prio['выгода'] = (prio.ожидаемо_удержим * prio.средняя_ценность).astype(int)
prio['затраты'] = (prio.клиентов * CONTACT_COST).astype(int)
prio['чистый_эффект'] = prio.выгода - prio.затраты
display(prio)
print()
print('Группы с отрицательным чистым эффектом из обработки стоит исключить —')
print('это освободит бюджет на те, где удержание окупается.')

### 15.4 Стоимость привлечения и удержания: расчёт под местный рынок

Публичных бенчмарков стоимости привлечения для банков Центральной Азии не существует — банки такие цифры не раскрывают. Заимствовать американские 55–70 долларов напрямую нельзя: разница в уровне зарплат и стоимости рекламы делает их неприменимыми.

Поэтому считаем снизу вверх, из проверяемых величин.

**Что берём за основу**

| Величина | Значение | Источник |
|---|---|---|
| Средняя зарплата по Узбекистану | 7,09 млн сум / мес | Нацкомстат, I полугодие 2026 |
| Средняя зарплата в Ташкенте | 12,01 млн сум / мес | там же |
| Зарплаты в банковском секторе | выше средних по стране | там же |
| Соотношение привлечение / удержание | 5–7× для банков | Bain & Company, отраслевые обзоры |
| Целевое отношение ценности к CAC | не ниже 3:1 | общепринятый ориентир |

**Стоимость контакта** выводится из стоимости рабочего времени оператора с учётом накладных расходов и доли недозвонов.

**Стоимость предложения** берётся прямо из данных: кампания `FEE_WAIVER_3M` — это три месяца без абонентской платы, а её размер известен из таблицы тарифных изменений.

**Стоимость привлечения** оценивается двумя независимыми способами: через соотношение 5–7× от стоимости удержания и через ограничение сверху, задаваемое отношением 3:1. Совпадение оценок — признак, что порядок величины угадан верно.

In [ ]:
# ---------- Стоимость контакта: расчёт снизу вверх ----------
AVG_SALARY_UZ = 7_090_000        # Нацкомстат, I полугодие 2026
AVG_SALARY_TASHKENT = 12_010_000

OPERATOR_SALARY = 6_000_000      # оператор колл-центра, ДОПУЩЕНИЕ (ниже средней)
OVERHEAD_MULT   = 2.2            # налоги, рабочее место, супервизия, обучение
WORK_HOURS_MON  = 168
MIN_PER_CONTACT = 10             # разговор плюс подготовка и пост-обработка
REACH_RATE      = 0.5            # доля дозвонов

fmt = lambda x: format(int(round(x)), ',d').replace(',', ' ')

hour_cost = OPERATOR_SALARY * OVERHEAD_MULT / WORK_HOURS_MON
call_cost = hour_cost / 60 * MIN_PER_CONTACT / REACH_RATE

print('Полная стоимость часа оператора : %s сум' % fmt(hour_cost))
print('Один результативный звонок      : %s сум' % fmt(call_cost))
print('  (%d мин работы при доле дозвона %.0f%%)' % (MIN_PER_CONTACT, 100 * REACH_RATE))
print()

CHANNEL_COST = {
    'call':  int(round(call_cost)),
    'sms':   150,        # рассылка плюс платформа
    'push':  10,
    'email': 20,
}
print('Стоимость доставки по каналам:')
for k, v in sorted(CHANNEL_COST.items(), key=lambda x: -x[1]):
    print('  %-6s %s сум' % (k, fmt(v)))

In [ ]:
# ---------- Стоимость предложения: из данных о тарифах ----------
fee = q("""SELECT avg(new_monthly_fee_uzs - old_monthly_fee_uzs) AS delta,
                  avg(new_monthly_fee_uzs) AS new_fee
           FROM public.churn_tariff_changes""")
MONTHLY_FEE = float(fee.new_fee[0]) if fee.new_fee[0] else float(fee.delta[0])
FEE_DELTA   = float(fee.delta[0])

# Кешбэк всегда ограничен потолком — без него расчёт даёт нереальные суммы.
# Потолок и срок действия — продуктовые параметры, ДОПУЩЕНИЕ.
CASHBACK_CAP    = 150_000     # максимум выплаты на клиента за всю акцию
CASHBACK_MONTHS = 3

med_month_turnover = float((test.sum_90.fillna(0) / 3).median())
print('Медианный месячный оборот клиента: %s сум' % fmt(med_month_turnover))
print('Абонентская плата после реформы  : %s сум/мес' % fmt(MONTHLY_FEE))
print('Прирост платы при реформе        : %s сум/мес' % fmt(FEE_DELTA))
print()

def cashback_cost(rate):
    """Кешбэк с потолком: без ограничения расчёт даёт нереалистичные суммы."""
    uncapped = med_month_turnover * rate * CASHBACK_MONTHS
    return min(uncapped, CASHBACK_CAP), uncapped

OFFER_COST = {'CALL_RETENTION': 0.0,
              'FEE_WAIVER_3M': MONTHLY_FEE * 3}
for code_, rate in [('WINBACK_10', 0.10), ('WINBACK_CASHBACK', 0.05)]:
    capped, uncapped = cashback_cost(rate)
    OFFER_COST[code_] = capped
    print('%-18s %s сум  (без потолка было бы %s)'
          % (code_, fmt(capped), fmt(uncapped)))
print('%-18s %s сум  (3 месяца абонплаты)' % ('FEE_WAIVER_3M', fmt(OFFER_COST['FEE_WAIVER_3M'])))
print('%-18s %s сум  (только разговор)' % ('CALL_RETENTION', 0))
print()
print('Потолок кешбэка %s сум — продуктовый параметр, требует подтверждения.' % fmt(CASHBACK_CAP))
if OFFER_COST['WINBACK_10'] == OFFER_COST['WINBACK_CASHBACK']:
    print()
    print('[!] Обе кешбэк-кампании упёрлись в потолок и стали неразличимы по цене.')
    print('    При таком потолке 10%% и 5%% дают одинаковую выплату, и кампании')
    print('    различаются только откликом. Если продукт задаёт разные потолки —')
    print('    подставьте их, иначе сравнение кампаний по стоимости теряет смысл.')

In [ ]:
# ---------- Полная цена контакта и отклика по кампаниям ----------
camp_cost = q("""SELECT campaign_code, channel, count(*) AS контактов,
                        sum(responded::int) AS откликнулись
                 FROM public.churn_retention_campaigns
                 GROUP BY 1, 2""")
camp_cost['доставка'] = camp_cost.channel.map(CHANNEL_COST)
camp_cost['предложение'] = camp_cost.campaign_code.map(OFFER_COST)
camp_cost['отклик_%'] = (100 * camp_cost.откликнулись / camp_cost.контактов).round(1)

# Доставка оплачивается всем контактам, предложение — только откликнувшимся.
camp_cost['цена_отклика'] = (
    (camp_cost.доставка * camp_cost.контактов +
     camp_cost.предложение * camp_cost.откликнулись) / camp_cost.откликнулись).round(0)
camp_cost['цена_контакта'] = (
    camp_cost.доставка + camp_cost.предложение * camp_cost.откликнулись
    / camp_cost.контактов).round(0)

# 95% интервал доли отклика — без него сравнение кампаний бессмысленно
from math import sqrt
z = 1.96
n_, x_ = camp_cost.контактов, camp_cost.откликнулись
pp = x_ / n_
se = np.sqrt(pp * (1 - pp) / n_)
camp_cost['отклик_ДИ'] = ['%.1f-%.1f%%' % (100 * max(a, 0), 100 * min(b, 1))
                          for a, b in zip(pp - z * se, pp + z * se)]

display(camp_cost.sort_values('цена_отклика')[
    ['campaign_code', 'channel', 'контактов', 'отклик_%', 'отклик_ДИ',
     'доставка', 'предложение', 'цена_контакта', 'цена_отклика']])

In [ ]:
# ---------- Различаются ли кампании на самом деле ----------
# Выбирать максимум из шумных оценок нельзя: при 16 комбинациях и ~200
# наблюдений в каждой самая высокая доля отклика почти всегда завышена.
tot_n = int(camp_cost.контактов.sum()); tot_x = int(camp_cost.откликнулись.sum())
p_pool = tot_x / tot_n
chi = float((((camp_cost.откликнулись - camp_cost.контактов * p_pool) ** 2)
             / (camp_cost.контактов * p_pool * (1 - p_pool))).sum())
dof = len(camp_cost) - 1
print('Общая доля отклика: %.1f%% (%d из %d)' % (100 * p_pool, tot_x, tot_n))
print('Хи-квадрат однородности: %.1f при %d степенях свободы' % (chi, dof))
crit = {15: 25.0, 14: 23.7, 13: 22.4, 12: 21.0}.get(dof, 25.0)
if chi < crit:
    print('Ниже критического значения %.1f — различия между кампаниями' % crit)
    print('статистически НЕ ЗНАЧИМЫ. Отклик везде примерно одинаков,')
    print('и выбирать кампанию по наблюдённой доле нельзя: это шум.')
    HOMOGENEOUS = True
else:
    print('Выше критического значения %.1f — кампании действительно различаются.' % crit)
    HOMOGENEOUS = False
print()

# Базовый сценарий: НЕ максимум отклика, а разумная кампания с предложением.
# CALL_RETENTION через push — доставка почти бесплатна и предложения нет;
# принимать её за эталон удержания нельзя, это уведомление, а не оффер.
CANDIDATES = camp_cost[camp_cost.предложение > 0]
base = CANDIDATES.loc[CANDIDATES.цена_отклика.idxmin()]

CONTACT_COST = float(base.цена_контакта)
RESPONSE_RATE = p_pool if HOMOGENEOUS else float(base['отклик_%']) / 100

print('Базовый сценарий: %s через %s' % (base.campaign_code, base.channel))
print('  предложение   : %s сум' % fmt(base.предложение))
print('  CONTACT_COST  : %s сум за контакт' % fmt(CONTACT_COST))
print('  RESPONSE_RATE : %.1f%% %s'
      % (100 * RESPONSE_RATE,
         '(общая доля — различия незначимы)' if HOMOGENEOUS else '(доля этой кампании)'))
print()
print('Почему не самая дешёвая строка таблицы: у CALL_RETENTION нет предложения,')
print('доставка через push стоит 10 сум, и наблюдённый отклик 17,3%% получен')
print('на 185 контактах. Принимать это за эталон удержания нельзя —')
print('уведомление без оффера не является программой удержания.')
print()
print('Диапазон по кампаниям с предложением: %s .. %s сум за контакт'
      % (fmt(CANDIDATES.цена_контакта.min()), fmt(CANDIDATES.цена_контакта.max())))
print('Выбор кампании — продуктовое решение: дорогое предложение может')
print('окупаться на ценных клиентах и не окупаться на массовых.')

In [ ]:
# ---------- Стоимость привлечения: три независимые оценки ----------
USD_UZS = 12_300
AVG_SALARY_US_USD = 5_000                      # порядок величины, для масштабирования
avg_salary_uz_usd = AVG_SALARY_UZ / USD_UZS
wage_ratio = AVG_SALARY_US_USD / avg_salary_uz_usd

# Способ 1 — потолок из отношения ценности к привлечению 3:1
clv_med = float(test_e.clv.median())
CAC_CEILING = clv_med / 3

# Способ 2 — международный ориентир, масштабированный по стоимости труда
CAC_SCALED_LOW  = 55 * USD_UZS / wage_ratio
CAC_SCALED_HIGH = 70 * USD_UZS / wage_ratio

# Способ 3 — от стоимости удержания через отраслевое соотношение 5-7x
CAC_RATIO_LOW  = CONTACT_COST * 5
CAC_RATIO_HIGH = CONTACT_COST * 7

print('Средняя зарплата: Узбекистан %s сум (%.0f долл.), США ~%s долл.'
      % (fmt(AVG_SALARY_UZ), avg_salary_uz_usd, fmt(AVG_SALARY_US_USD)))
print('Соотношение стоимости труда: %.1fx' % wage_ratio)
print()
display(pd.DataFrame([
    {'способ': '1. Потолок при отношении ценности к CAC 3:1',
     'оценка, сум': fmt(CAC_CEILING)},
    {'способ': '2. Международный ориентир 55-70 долл., масштабированный по труду',
     'оценка, сум': '%s .. %s' % (fmt(CAC_SCALED_LOW), fmt(CAC_SCALED_HIGH))},
    {'способ': '3. От стоимости удержания (соотношение 5-7x)',
     'оценка, сум': '%s .. %s' % (fmt(CAC_RATIO_LOW), fmt(CAC_RATIO_HIGH))},
]))

# Принимаем середину пересечения способов 1 и 2 — они независимы
CAC_UZS = float(np.mean([CAC_SCALED_LOW, CAC_SCALED_HIGH]))
print()
print('Принимаем CAC = %s сум (~%.0f долл.)' % (fmt(CAC_UZS), CAC_UZS / USD_UZS))
print()
print('Обоснование выбора: способы 1 и 2 независимы — один опирается на ценность')
print('клиента из наших данных, другой на международный ориентир, приведённый')
print('к местной стоимости труда. Их согласие подтверждает порядок величины.')
print('Способ 3 чувствителен к выбору кампании: при дорогом предложении')
print('стоимость удержания раздувается, и оценка теряет смысл.')
print()
if CAC_UZS <= CAC_CEILING:
    print('CAC укладывается в потолок 3:1 — экономика привлечения сходится.')
    print('Запас: ценность клиента выше стоимости привлечения в %.1f раза.'
          % (clv_med / CAC_UZS))
else:
    print('[!] CAC выше потолка 3:1 при марже %.0f%% и горизонте %d мес.'
          % (100 * MARGIN_RATE, HORIZON_MONTHS))
    print('    Проверьте эти допущения: реальный срок жизни клиента в банке')
    print('    обычно измеряется годами, а не одним годом.')

In [ ]:
# --- чувствительность к допущениям
sens = []
for margin in [0.01, 0.02, 0.04]:
    for cost in [5000, 15000, 40000]:
        clv = mon_turnover * margin * HORIZON_MONTHS
        te = test.assign(p=p_te, clv=clv).sort_values('p', ascending=False)
        best_eff, best_share = None, None
        for share in [0.05, 0.10, 0.20, 0.30, 0.50, 1.00]:
            k = int(len(te) * share); sel = te.head(k)
            eff = sel[sel.target == 1].clv.sum() * RESPONSE_RATE - k * cost
            if best_eff is None or eff > best_eff:
                best_eff, best_share = eff, int(share * 100)
        sens.append({'маржа': margin, 'стоимость контакта': cost,
                     'оптимум_%': best_share, 'эффект': int(best_eff)})
display(pd.DataFrame(sens).pivot(index='маржа', columns='стоимость контакта',
                                 values='оптимум_%'))
print('Таблица показывает, какую долю списка обрабатывать при разных допущениях.')
print('Если оптимум устойчив в широком диапазоне — рекомендация надёжна.')

## 16. Что предлагать по топу риска

Одинаково обрабатывать всех из топа неэффективно: причины ухода разные, и цена удержания разная.

- **Технический отток.** Клиенты с неудачными списаниями (`chg_failed_90`) уходят не по своей воле: не прошёл платёж, истекла карта. Самая дешёвая группа — достаточно напоминания об обновлении реквизитов.
- **Тарифная чувствительность.** Затронутые повышением уходят в 1,6 раза чаще. Для них работает адресное предложение, а не общая скидка.
- **Затухание активности.** Падающий `trend_30_90` при сохранённой истории — клиент постепенно перестаёт пользоваться. Здесь нужен продуктовый стимул, и результат наименее предсказуем.
- **Недовольство сервисом.** Нерешённые обращения и низкая оценка — сначала решить проблему, потом предлагать.

Порядок обработки задаётся не только вероятностью ухода, но и ценностью клиента: `p × clv` даёт ожидаемые потери в сумах, и сортировать список правильнее по ним.

In [ ]:
seg = test_e.copy()
seg['ожидаемые_потери'] = seg.p * seg.clv

def group(r):
    if r.get('chg_failed_90', 0) > 0: return 'технический отток'
    if r.get('tickets_unresolved_180', 0) > 0: return 'недовольство сервисом'
    if pd.notna(r.get('trend_30_90')) and r.get('trend_30_90', 1) < 0.6: return 'затухание активности'
    return 'прочие'

top = seg.sort_values('ожидаемые_потери', ascending=False).head(int(len(seg) * 0.2)).copy()
top['группа'] = top.apply(group, axis=1)
display(top.groupby('группа').agg(
    клиентов=('target', 'size'),
    фактический_отток_pct=('target', lambda s: round(100 * s.mean(), 2)),
    средние_потери=('ожидаемые_потери', lambda s: int(s.mean()))))

print()
print('Сравнение двух способов упорядочить список:')
by_p = seg.sort_values('p', ascending=False).head(1000)
by_l = seg.sort_values('ожидаемые_потери', ascending=False).head(1000)
print('  по вероятности ухода : ушли %d, сумма потерь %s'
      % (by_p.target.sum(), format(int(by_p[by_p.target == 1].clv.sum()), ',d').replace(',', ' ')))
print('  по ожидаемым потерям : ушли %d, сумма потерь %s'
      % (by_l.target.sum(), format(int(by_l[by_l.target == 1].clv.sum()), ',d').replace(',', ' ')))
print('Второй способ ловит меньше клиентов, но дороже стоящих.')

---
# 17. Насколько модели можно верить

Три проверки. Каждая отвечает на конкретное сомнение, и каждая может дать неприятный ответ — в этом и смысл.

1. **Воспроизводится ли результат на другом периоде.** Сейчас проверка одна: обучение на июне, проверка на сентябре. Одна пара снимков ничего не говорит об устойчивости.
2. **Почему метрика на проверке выше, чем на обучении.** Разрыв −0,065 у регрессии противоречит здравому смыслу: модель подгоняется под обучающие данные и должна показывать там результат выше.
3. **Сколько признаков нужно на самом деле.** Если 51 признак выигрывает у трёх на сотые доли, честнее сказать, что задача решается тремя.

### 17.1 Второй проверочный период

Схема проверки: обучение на **самом раннем** снимке (31 марта), проверка на каждом последующем — на июньском и на сентябрьском. Три снимка в конфиге уже есть, отдельная пересборка не нужна: окно таргета для мартовского снимка закрывается 29 июня, внутри данных.

**Что именно здесь проверяется.** У мартовского снимка окно исхода с 30 апреля по 29 июня — целиком до тарифной реформы. Если гипотеза из 17.2 верна, модель, обученная на спокойном периоде, покажет заметно разные результаты на июньском снимке (реформа внутри окна) и на сентябрьском (реформа уже прошла). Разброс между двумя проверочными периодами и есть ответ на вопрос, какую цифру можно заявлять.

Если метрика держится на обоих — результат устойчив, называем одно число. Если гуляет — называем диапазон и объясняем причину. Второе не хуже первого: диапазон с объяснением защищается лучше, чем одна цифра без него.

In [ ]:
snaps = sorted(full.snapshot_label.unique())
print('Доступные снимки:', snaps)

if len(snaps) < 3:
    print()
    print('Третьего снимка нет. Добавьте S0_early в конфиг ноутбука сборки')
    print('и перезапустите его, иначе эта проверка невозможна.')
else:
    tr0 = full[full.snapshot_label == snaps[0]]
    rows = []
    for kind in ['logreg', 'hgb']:
        mm, cc = build(FEATS, kind)
        mm.fit(tr0[cc], tr0.target)
        r = {'модель': kind,
             'Gini на обучении (%s)' % snaps[0]:
                 round(gini(tr0.target, mm.predict_proba(tr0[cc])[:, 1]), 3)}
        for s in snaps[1:]:
            te = full[full.snapshot_label == s]
            r['Gini %s' % s] = round(gini(te.target, mm.predict_proba(te[cc])[:, 1]), 3)
        rows.append(r)
    display(pd.DataFrame(rows))
    print()
    print('Разброс между проверочными периодами показывает, насколько результат')
    print('зависит от конкретного среза. Большой разброс означает, что модель')
    print('требует частого переобучения, и одну цифру заявлять нельзя.')

### 17.2 Откуда отрицательный разрыв

Отрицательный разрыв есть только у логистической регрессии (Gini на обучении 0,581 против 0,624 на проверке). У бустинга разрыв обычный, положительный, +0,070. Это сразу сужает круг объяснений: дело не в схеме измерения в целом, иначе перекосило бы обе модели.

**Основная гипотеза: обучающий период объективно труднее.** Обучающий снимок — 30 июня, его окно исхода с 30 июля по 28 сентября, то есть целиком после тарифной реформы 1 июля. Проверочный снимок — 30 сентября, окно исхода с 30 октября по 29 декабря, когда всплеск уже прошёл. Это видно по долям оттока: 7,31% в обучении против 5,44% в проверке. Лишний отток в обучении вызван причиной, которой в признаках нет и быть не может — тарифные признаки в обучающем срезе имеют нулевую дисперсию и отсеяны в разделе 10.1. Модель не в состоянии ранжировать людей, ушедших по невидимой ей причине, и метрика на обучении проседает.

**Побочная гипотеза: оценка на обучении занижена самим методом.** Регрессия учится с балансировкой классов и сильной регуляризацией; in-sample метрика при этом может оказаться ниже, чем на выборке с другим соотношением классов. Это объясняет часть разрыва, но не объясняет, почему у бустинга разрыв нормальный.

Различаем три способами: честной кросс-валидацией внутри обучающего снимка, состязательной проверкой (насколько легко отличить обучающую выборку от проверочной) и сравнением распределений признаков между снимками.

**Если основная гипотеза верна**, состязательная проверка покажет заметный сдвиг популяции, а кросс-валидация внутри обучения окажется ближе к 0,58, чем к 0,62. Формулировка для защиты тогда такая: модель обучена на периоде с разовым шоком, которого она не видит, поэтому её собственная оценка качества на этом периоде занижена. Заявлять надо диапазон по нескольким периодам, а не одну цифру.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

# --- 1. честная оценка внутри обучающего снимка
m_cv, cc = build(FEATS, 'logreg')
cv = StratifiedKFold(5, shuffle=True, random_state=42)
sc = cross_val_score(m_cv, train[cc], train.target, cv=cv, scoring='roc_auc')
print('Кросс-валидация внутри обучающего снимка: Gini %.3f ± %.3f'
      % (2 * sc.mean() - 1, 2 * sc.std()))
print('Gini in-sample на обучении : %.3f' % results[0]['Gini train'])
print('Gini на проверочном снимке : %.3f' % results[0]['Gini test'])
print()
print('Если кросс-валидация близка к проверке, то заниженной была оценка')
print('in-sample, а не завышенной — проверочная.')
print()

# --- 2. состязательная проверка: отличима ли проверка от обучения
adv = pd.concat([train.assign(_is_test=0), test.assign(_is_test=1)], ignore_index=True)
m_adv, cc_adv = build(FEATS, 'logreg')
sc_adv = cross_val_score(m_adv, adv[cc_adv], adv['_is_test'], cv=3, scoring='roc_auc')
print('Состязательная проверка: AUC %.3f (Gini %.3f)'
      % (sc_adv.mean(), 2 * sc_adv.mean() - 1))
print('Около 0,5 — выборки неотличимы, распределения совпадают.')
print('Заметно выше — популяция между снимками сдвинулась.')
print()

# --- 3. что именно сдвинулось
uni_df = uni if isinstance(uni, pd.DataFrame) else pd.DataFrame(uni)
uni_df = uni_df.sort_values('одномерный_Gini', ascending=False)
top_feats = uni_df.head(10).признак.tolist()
cmp_dist = pd.DataFrame({
    'обучение (медиана)': train[top_feats].median().round(2),
    'проверка (медиана)': test[top_feats].median().round(2),
})
cmp_dist['изменение_%'] = (100 * (cmp_dist['проверка (медиана)'] /
                                  cmp_dist['обучение (медиана)'].replace(0, np.nan) - 1)).round(1)
display(cmp_dist)

# --- 4. насколько "выразительны" ушедшие в каждом снимке
sep = []
for lb in sorted(full.snapshot_label.unique()):
    d = full[full.snapshot_label == lb]
    sep.append({'снимок': lb,
                'медиана cnt_30 у оставшихся': round(d[d.target == 0].cnt_30.median(), 1),
                'медиана cnt_30 у ушедших': round(d[d.target == 1].cnt_30.median(), 1),
                'разрыв': round(d[d.target == 0].cnt_30.median()
                                - d[d.target == 1].cnt_30.median(), 1)})
display(pd.DataFrame(sep))
print('Чем больше разрыв между классами, тем легче выборка для модели.')
print()
print('--- Как читать результат ---')
print('Три оценки одной модели:')
print('  кросс-валидация внутри обучения — честная оценка на «своём» периоде')
print('  in-sample на обучении           — завышена подгонкой, но занижена регуляризацией')
print('  на проверочном снимке           — зависит от того, насколько период похож')
print()
print('Если проверочная выше кросс-валидации, а состязательная проверка заметно')
print('выше 0,5 — проверочный период просто легче. Тогда заявлять его метрику')
print('как ожидаемое качество нельзя: нужна оценка на нескольких периодах,')
print('а осторожный ориентир даёт кросс-валидация.')

**Что именно делает выборки различимыми.** Состязательная проверка показала AUC 0,740 — формально это заметный сдвиг популяции. Но медианы ключевых признаков между снимками почти не изменились (`cnt_30` 6 против 6, `ev_30` 8 против 8). Значит различие создаётся не активностью, и прежде чем заявлять «популяция сдвинулась», надо посмотреть, какой признак его создаёт. Между снимками 90% одних и тех же клиентов, ставших на 92 дня старше, — если сверху окажется стаж или другой календарный признак, сдвиг механический и содержательного смысла не несёт.

In [ ]:
# Одномерная состязательная сила: насколько каждый признак сам по себе
# отличает обучающую выборку от проверочной.
adv_rows = []
lab = np.r_[np.zeros(len(train)), np.ones(len(test))]
for c in FEATS:
    if not pd.api.types.is_numeric_dtype(full[c]):
        continue
    v = pd.concat([train[c], test[c]], ignore_index=True)
    m_ = v.notna().values
    if m_.sum() < 100 or v[m_].nunique() < 2:
        continue
    a = roc_auc_score(lab[m_], v[m_].values)
    adv_rows.append({'признак': c, 'AUC обучение против проверки': round(max(a, 1 - a), 3)})

adv_df = (pd.DataFrame(adv_rows)
          .sort_values('AUC обучение против проверки', ascending=False)
          .reset_index(drop=True))
display(adv_df.head(12))

top_adv = adv_df.iloc[0]
print()
print('Сильнее всего разделяет: %s (AUC %.3f)'
      % (top_adv['признак'], top_adv['AUC обучение против проверки']))
print()
print('Как читать. Если сверху стоят стаж, возраст или другие признаки, которые растут')
print('у всех одинаково просто потому, что снимки отстоят на 92 дня, — сдвиг')
print('механический. Он не означает, что клиенты стали другими, и ссылаться на него')
print('как на смену популяции нельзя.')
print('Если же сверху признаки активности или сумм — сдвиг содержательный, и модель')
print('действительно требует регулярного переобучения.')


### 17.3 Сколько признаков нужно на самом деле

Лестница: одна фича, три, десять, все. Если полный набор выигрывает у трёх на сотые доли, то заявлять 51 признак как достижение нечестно — задача решается тремя, а остальное усложняет модель без пользы.

Отбор идёт по одномерной силе, посчитанной **только на обучающем снимке**.

In [ ]:
ladder = []
uni_df = uni if isinstance(uni, pd.DataFrame) else pd.DataFrame(uni)
uni_df = uni_df.sort_values('одномерный_Gini', ascending=False)
ranked = uni_df.признак.tolist()
for n in [1, 3, 5, 10, 20, len(FEATS)]:
    cols_use = ranked[:n] if n < len(FEATS) else FEATS
    mm, cc = build(cols_use, 'logreg')
    mm.fit(train[cc], train.target)
    ladder.append({'признаков': len(cols_use),
                   'какие': ', '.join(cols_use[:3]) + ('...' if len(cols_use) > 3 else ''),
                   'Gini test': round(gini(test.target,
                                           mm.predict_proba(test[cc])[:, 1]), 3)})
lad = pd.DataFrame(ladder)
display(lad)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(lad.признаков, lad['Gini test'], marker='o')
ax.set_xscale('log'); ax.set_xlabel('число признаков (лог. шкала)')
ax.set_ylabel('Gini на проверке'); ax.set_title('Сколько признаков реально нужно')
plt.tight_layout(); plt.show()

gain = lad['Gini test'].iloc[-1] - lad['Gini test'].iloc[1]
print('Переход от 3 признаков ко всем даёт %+.3f Gini.' % gain)
if gain < 0.02:
    print('Прирост незначителен: задача решается тремя признаками, и честнее')
    print('заявлять компактную модель, а не полный набор.')

### 17.4 Вывод о качестве

**Собранные оценки одной и той же модели:**

| Что измеряли | Gini |
|---|---|
| Кросс-валидация внутри обучающего снимка (июнь) | 0,520 ± 0,034 |
| In-sample на обучении (июнь) | 0,581 |
| Проверка на сентябрьском снимке | 0,624 |
| Обучение на марте → проверка на июне | 0,526 |
| Обучение на марте → проверка на сентябре | 0,622 |

**Главный вывод: неустойчива не модель, а период.** Две независимые оценки качества на июньском периоде — кросс-валидация внутри него (0,520) и внешняя проверка моделью, обученной на марте (0,526) — совпали. Две оценки на сентябрьском периоде тоже совпали: 0,624 и 0,622. Разброс между периодами 0,10, разброс между способами измерения внутри периода — 0,006.

Причина разницы между периодами названа в 17.2 и подтверждается разрывом классов: медиана `cnt_30` у оставшихся против ушедших даёт 4,0 на мартовском снимке, **3,0 на июньском** и 4,0 на сентябрьском. Июньская выборка объективно труднее — ушедшие там меньше отличаются по активности, потому что часть из них ушла из-за тарифа, а не из-за затухания.

**Что заявлять.** Не одну цифру, а две с объяснением:
- **0,62** — рабочее качество на спокойном периоде;
- **0,52** — нижняя граница на периоде с разовым внешним событием.

Разовый шок вроде тарифной реформы моделью не покрывается по построению: в обучающем срезе таких признаков нет ни у кого. Он требует отдельного сценарного расчёта и переобучения после события. Это ограничение дизайна, названное заранее, а не провал.

**Сколько признаков нужно.** Лестница не выходит на плато: 1 признак даёт 0,507, три — 0,524, двадцать — 0,540, все 52 — 0,624. Переход от трёх ко всем даёт **+0,100 Gini**, причём основной прирост (+0,084) появляется на последнем шаге, от 20 признаков ко всем. Полный набор оправдан, компактную модель заявлять не нужно.

Отдельно стоит отметить, что ранжирование по одномерному Gini плохо предсказывает полезность: признаки, слабые поодиночке, дают наибольший прирост в наборе. Это нормально для линейной модели с категориальными переменными и стоит того, чтобы сказать вслух, если спросят про отбор признаков.

**Выбор модели.** Бустинг выиграл на одной паре снимков (0,651 против 0,624, парный бутстрэп дал интервал [+0,005; +0,049], ноль вне интервала). Но при обучении на мартовском снимке регрессия впереди на **обоих** проверочных периодах: 0,526 против 0,513 и 0,622 против 0,603, при Gini на обучении 0,960 — то есть бустинг запоминает обучающий снимок. Преимущество не воспроизвелось, выбрана логистическая регрессия.

Это отдельный сюжет, который стоит рассказать: разница была статистически значима по бутстрэпу на одной выборке и всё равно оказалась свойством периода. Значимость и воспроизводимость — разные вещи.

**Про состязательную проверку.** AUC 0,740 выглядит как сильный сдвиг популяции, но одномерная диагностика показывает другое: ни один признак сам по себе не различает периоды сильнее AUC 0,581. Сверху стоят `products_ever` (0,581), `products_open` (0,580), `cards_cnt` (0,573) — кумулятивные счётчики, которые растут у всех просто с течением времени, — и тренды `trend_30_90` (0,579), `trend_90_180` (0,569), отражающие сезонность лето против осени.

То есть 0,740 набирается из множества мелких календарных различий, а не из смены клиентской базы. Уровень активности между снимками не изменился вовсе: медианы `cnt_30` 6 против 6, `ev_30` 8 против 8, `app_recency_days` 3 против 3.

**Следствие для защиты:** состязательная проверка не объясняет разрыв 0,52 против 0,62. Календарный дрейф есть, но он одинаково затрагивает оба класса. Разрыв объясняется тарифной реформой внутри окна исхода июньского снимка — это подтверждается разрывом классов по `cnt_30` (3,0 на июне против 4,0 на марте и сентябре), а не составом выборки.

## 18. Итоги для защиты

> Числа модели и таргета ниже — с прогона на текущем конфиге (порог тишины 60 дней, лаг 30 дней).
> Блоки «Экономика» и «Uplift» помечены отдельно: их надо сверить с выводами разделов 12.3 и 15
> после перепрогона, там ещё стоят значения с прежней версии таргета.

### Часть 1. Таргет

**Определение.** Клиент считается ушедшим, если не совершил ни одной успешной операции в окне исхода длиной 60 дней, начинающемся через 30 дней после даты снимка.

```
[ ---- признаки ---- ] t0 [ -- лаг 30 дн. -- ][ -- окно исхода 60 дн. -- ]
```

**Зачем лаг.** Без него модель предсказывает то, на что уже нельзя повлиять. Лаг — время на скоринг, решение и контакт.

**Главный аргумент, что таргет не сводится к давности** — три числа:

| Проверка | Результат |
|---|---|
| Доля ушедших, активных внутри лага | 45,4% / 44,8% / 36,7% по снимкам |
| Медиана дней молчания на дату снимка | 8 у ушедших против 4 у оставшихся |
| Цена снятия всех признаков давности | −0,001 Gini (0,624 → 0,623) |

Почти половина ушедших на дату снимка выглядела активной. Одна давность в одиночку даёт Gini 0,425 против 0,624 у полного набора — вклад есть, монополии нет. На прежней версии таргета без лага та же проверка давала 0,724 из 0,846, то есть модель почти целиком стояла на давности. Лаг эту опору убрал, и метрика закономерно упала.

**Порог тишины 60 дней** выбран по кривой реактивации: там, где доля возвращающихся выходит на плато.

**Исключены из базы:** зарегистрированные после снимка, без единой операции, уже молчавшие дольше порога, незрелые со стажем менее 180 дней.

**Три снимка и доли оттока:**

| Снимок | Дата | Клиентов | Ушли | Доля |
|---|---|---|---|---|
| S0_early | 31.03.2025 | 9 817 | 581 | 5,92% |
| S1_train | 30.06.2025 | 10 463 | 765 | 7,31% |
| S2_test | 30.09.2025 | 10 865 | 591 | 5,44% |

Неоднозначных случаев нет.

**Причина всплеска в июне** — тарифная реформа 1 июля 2025: 8 044 клиента, все изменения в сторону повышения, в среднем +15 654 сума в месяц. Затронутые уходят чаще на 4,19 п.п., z = 9,75, интервал [+3,29; +5,09]. Группы при этом неразличимы по сегменту, региону, каналу, стажу и активности — значит эффект не объясняется составом групп.

Всплеск виден именно в S1_train, потому что его окно исхода (30 июля — 28 сентября) целиком лежит после реформы. У S2_test окно с 30 октября по 29 декабря, там всплеск уже прошёл.

**Годовой отток заметно выше рыночных 13–19%.** Разница не в качестве портфеля, а в определении: рынок считает закрытие счёта, мы — прекращение активности. Это «тихий» отток, который не виден в отчётности, но на него ещё можно повлиять.

### Часть 2. Модель

**Признаки:** 52 из 64. Отсеяны 12 с нулевой дисперсией в обучении, включая все тарифные (`tariff_changes_cnt`, `days_since_tariff_change`, `tariff_changed_90d`). Разрывов покрытия между снимками нет.

**Почему тарифные признаки исключены.** Реформа вступила в силу 1 июля, обучающий снимок — 30 июня. В обучении о ней нет ни одной записи. Модель, обученная на июне, физически не может научиться на неё реагировать. Оставить признаки — значит создать ложное впечатление, что модель их учла.

**Валидация по времени**, не случайная. Признаки строго до `t0`.

**Выбранная модель — логистическая регрессия.** Обоснование не в метрике на одной паре снимков, а в воспроизводимости:

| Модель | Обучение на марте | Проверка на июне | Проверка на сентябре |
|---|---|---|---|
| Логистическая регрессия | 0,647 | **0,526** | **0,622** |
| Градиентный бустинг | 0,960 | 0,513 | 0,603 |

Бустинг выиграл при обучении на июне (0,651 против 0,624), парный бутстрэп дал интервал [+0,005; +0,049], ноль вне интервала. Но при смене обучающего периода преимущество исчезло, а Gini 0,960 на обучении показывает, что бустинг запоминает снимок. **Статистическая значимость на одной выборке и воспроизводимость — разные вещи.**

**Какое качество заявляем — диапазон, а не одну цифру:**

- **Gini 0,62** — на спокойном периоде;
- **Gini 0,52** — на периоде с разовым внешним событием.

Обоснование диапазона: две независимые оценки на июньском периоде (кросс-валидация внутри него 0,520 ± 0,034 и внешняя проверка марта 0,526) совпали, две оценки на сентябрьском (0,624 и 0,622) тоже совпали. Разброс между способами измерения внутри периода 0,006, между периодами — 0,10. Неустойчива не модель, а период.

Причина видна по разрыву классов: медиана `cnt_30` у оставшихся против ушедших даёт 4,0 на марте, 3,0 на июне, 4,0 на сентябре. Июньская выборка объективно труднее.

**Отрицательный разрыв обучение-проверка** (0,581 против 0,624) объясняется тем же: модель обучена на периоде с шоком, которого не видит, поэтому её оценка на собственном периоде занижена. При обучении на мартовском снимке разрыв нормальный, положительный: 0,647 против 0,622.

**Метрики на проверочном снимке:**

| Метрика | Значение |
|---|---|
| ROC-AUC | 0,812 |
| Gini | 0,624 |
| KS | 0,472 |
| PR-AUC | 0,366 при базовом уровне 0,054 |
| На 5% списка | Precision 0,401, Recall 0,369, F1 0,384 |

Порог 0,5 не используется: при доле оттока 5,4% он бессмысленен — Precision падает до 0,139. Рабочий порог задаёт экономика. На 5% списка точность в 7,4 раза выше случайного отбора.

**Сколько признаков нужно.** Лестница не выходит на плато: один признак 0,507, три 0,524, двадцать 0,540, все 52 — 0,624. Переход от трёх ко всем даёт +0,100 Gini, причём основной прирост (+0,084) появляется на последнем шаге, от 20 признаков ко всем. Полный набор оправдан, компактную модель заявлять не нужно.

Прирост дают не демографические признаки — их вклад отдельно измерен и равен +0,006 (раздел 12.4). Его дают многочисленные слабые поведенческие признаки. Отсюда вывод, который стоит сказать вслух: **ранжирование по одномерному Gini плохо предсказывает полезность признака в наборе**.

**Демография.** Только поведение (45 признаков) — 0,618, только демография (7) — 0,284, вместе — 0,624. Зарплатный признак значим сам по себе: отток 3,23% против 9,03%, z = 10,41, одномерный Gini 0,178. Но в присутствии поведения демография добавляет 0,006 — она не шум, а дублирующая информация. В продуктовой версии её следует убрать: `gender`, `region` и `age_years` среди предикторов создают регуляторный риск, несоразмерный 0,006 Gini.

**Подход** — комбинация бинарной классификации и анализа выживаемости. Классификация даёт список приоритетов, выживаемость добавляет измерение времени и учитывает цензурированные наблюдения.

### Uplift

**Главный результат раздела — отрицательный, и это содержательный ответ.** Эффект удержания на этих данных не идентифицируется.

Воздействием взят контакт внутри лага: признаки до него, исход после — причинная последовательность соблюдена. Дальше проверялись условия причинного вывода, и они не выполнились:

| Группа | Контактов | Осталось |
|---|---|---|
| Возврат (`WINBACK_*`) | 103 | 0,00% |
| Удержание (`CALL_RETENTION`, `FEE_WAIVER_3M`) | 101 | 4,95% |
| Контроль | 30 941 | 94,38% |

Ноль из 103 у возвратных кампаний — это определение, а не эффект: возврат применяют к уже ушедшим. Прямое доказательство назначения по исходу — медианная давность последней операции: **4 дня у контроля против 34 дней у контактированных**. Банк связывается с теми, кто уже месяц молчит; при пороге тишины 60 дней такой клиент попадёт в отток почти независимо от контакта.

**Формальные числа.** Модель склонности отличает контактированных с AUC 0,940 — назначение почти детерминированное. Проверка общей опоры по границам диапазона формально проходит (склонность контроля от 0,0004 до 0,802 накрывает все 204 контакта), но это ловушка: она сравнивает границы, а не плотности. Эффективный размер выборки у контактированных 23,6 при n = 204, то есть 11,6%, и одно наблюдение забирает 12,6% веса. Взвешенная оценка опиралась бы на два десятка человек.

Нарушено условие позитивности, а не только сопоставимости, — поэтому не спасают ни взвешивание, ни AIPW, ни двойная устойчивость.

**Формулировка для защиты:** сценарий «не взаимодействуем» не наблюдаем для тех, с кем взаимодействовали. Наблюдаемая разница измеряет критерий отбора банка, а не пользу контакта.

**Что из этого следует для бизнеса** (раздел 12.3.8): программа удержания запускается слишком поздно — через 34 дня молчания при лаге на вмешательство 30 дней. Она работает как возврат, а называется удержанием. Первая рекомендация — сдвинуть момент контакта к дате скоринга. Вторая — эксперимент со случайной контрольной группой и флагом вмешательства в витрине; расчёт размера выборки приведён там же.

**Все шестнадцать комбинаций кампания-канал дали статистически неразличимый отклик:** хи-квадрат 22,3 при критическом 25,0, общая доля 12,6% на 3 305 контактах, разброс от 8,1% до 17,3% при выборках примерно по 200 контактов. Пока различия не доказаны, кампанию выбирают по стоимости, а не по наблюдаемому отклику.

### Экономика

> Числа сверить с разделами 15 и 15.4 после перепрогона.

Все параметры выведены снизу вверх под местный рынок: стоимость часа оператора из зарплаты и накладных, стоимость контакта из доли дозвона, стоимость предложения из данных о тарифах, стоимость привлечения тремя независимыми методами. Международные бенчмарки использованы только как контекст.

**Ограничение, которое надо назвать самому.** Доля отклика 12,6% входит в расчёт как эффект удержания, хотя это доля отклика на кампанию. Равенство «откликнулся = остался» на этих данных не проверяется — см. блок про uplift выше. Поэтому экономика раздела 15 — это **оценка потенциала при явно названном допущении**, а не измеренная выгода. Заменить допущение на число можно только экспериментом из 12.3.8.

### Что назвать самому, до того как спросят

- Данные синтетические, зависимости в них чище реальных.
- Качество заявлено диапазоном 0,52–0,62, потому что периоды различаются между собой сильнее, чем способы измерения внутри периода.
- Разовый шок вроде тарифной реформы моделью не покрывается по построению и требует переобучения после события, а не заранее.
- Ранжирование признаков по одномерному Gini плохо предсказывает их полезность в наборе: основной прирост дают признаки, слабые поодиночке.
- Состязательная проверка даёт AUC 0,740, но ни один признак поодиночке не различает периоды сильнее 0,581 — это календарный дрейф кумулятивных счётчиков и сезонность, а не смена клиентской базы.
- Демография в модели оставлена, хотя её вклад 0,006; в продуктовой версии её надо убрать.
- Все параметры экономики — допущения, требующие подтверждения продуктовым и финансовым блоком: потолок кешбэка, стоимость каналов, маржа, горизонт.
- Эффект удержания на этих данных не просто «оценка вместо измерения» — он не идентифицируется в принципе, потому что контакт назначается по признаку, который сам предсказывает исход. Дизайн эксперимента и расчёт размера выборки — в разделе 12.3.8.
- Экономика раздела 15 приравнивает отклик к удержанию. Это допущение, а не результат.
- Анализ выживаемости требовал аккуратного цензурирования: наивная версия давала 100% событий и бессмысленные коэффициенты Кокса. Исправленная версия даёт 16,8% событий за 184 дня и 83,2% цензурированных.

---

## 19. Ответы на вопросы формы сдачи (Q1–Q6)

Q1, Q2 и Q3 требуют конкретных чисел из базы. Ячейка ниже считает их одним запросом, чтобы на защите не пересчитывать. Q4–Q6 — текстовые, их формулировки собраны под ячейкой.

In [ ]:
# ---------- Q1. Сколько всего клиентов ----------
q1 = q("SELECT count(*) AS всего_клиентов FROM public.churn_customers")
display(q1)

# ---------- Q2. Уникальные клиенты с успешной операцией в декабре 2025 ----------
q2 = q("""
    SELECT count(DISTINCT customer_id) AS клиентов_с_операцией_дек2025
    FROM %(txn)s t
    WHERE txn_ts >= DATE '2025-12-01' AND txn_ts < DATE '2026-01-01'
      %(succ)s
""" % dict(txn=TXN, succ=SUCC))
display(q2)

# ---------- Q3. Охват тарифных изменений ----------
q3 = q("""
    SELECT count(*) AS всего_изменений,
           count(DISTINCT customer_id) AS затронуто_клиентов,
           count(*) FILTER (WHERE new_monthly_fee_uzs > old_monthly_fee_uzs) AS повышений,
           round(avg(new_monthly_fee_uzs - old_monthly_fee_uzs)) AS средняя_дельта,
           min(effective_date)::date AS первое,
           max(effective_date)::date AS последнее
    FROM public.churn_tariff_changes
""")
display(q3)

n_all = int(q1.iloc[0, 0])
n_aff = int(q3.iloc[0, 1])
print('Q1: клиентов в базе — %s' % format(n_all, ',d').replace(',', ' '))
print('Q2: уникальных клиентов с успешной операцией в декабре 2025 — %s'
      % format(int(q2.iloc[0, 0]), ',d').replace(',', ' '))
print('Q3: тарифные изменения затронули %s клиентов — %.1f%% базы'
      % (format(n_aff, ',d').replace(',', ' '), 100 * n_aff / n_all))
print('    все изменения — в сторону повышения: %d из %d'
      % (int(q3.iloc[0, 2]), int(q3.iloc[0, 0])))
print('    средний прирост платы: %s сум/мес' % format(int(q3.iloc[0, 3]), ',d').replace(',', ' '))
print()
print('Оговорка к Q2: считаются успешные операции (status = %r) в churn_transactions,' % SUCCESS_VALUE)
print('архив не подключается — он полный дубликат основной таблицы (раздел 0.3).')


### Q4. Какие таблицы не использовали и почему

**Содержат информацию из будущего — это отдельная категория:**

- `churn_customers_360` — колонки `is_churned`, `churn_flag_date`, `days_since_last_txn`, `total_txn_cnt` посчитаны на дату выгрузки. Это готовый таргет и агрегаты, собранные после момента предсказания. Использование любой из них означало бы прямую утечку.
- `churn_nps_survey` — сами данные корректны, но опросы проводились с 1 ноября по 31 декабря 2025, то есть после всех трёх снимков. Признак, которого в момент предсказания ещё не существует, использовать нельзя.

**Не использованы по другим причинам:**

- `churn_transactions_archive` — полный дубликат: все 815 568 строк уже есть в основной таблице (проверено по `txn_id`), плюс внутри самого архива 31 093 задвоенных операции. Подключение только удвоило бы часть истории.
- `churn_customers_crm_old` — устаревшая копия справочника: ключ `client_id` вместо `customer_id`, 14 000 строк против 20 000.
- `churn_temp` — служебная, пустая.
- `churn_retention_campaigns` — **не в признаки**, но использована отдельно. Контакт с клиентом есть следствие оценки риска банком, то есть признак, порождённый решением о самом клиенте. В модели риска это утечка; в разделе 12.3 эта же таблица используется по назначению — как воздействие в задаче причинного вывода.

### Q5. Защита таргета

Клиент считается ушедшим, если не совершил ни одной успешной операции в окне исхода длиной 60 дней, начинающемся через 30 дней после даты снимка.

Три аргумента, что это не пересказ давности последней операции:

1. **45,4% / 44,8% / 36,7%** ушедших по снимкам были активны внутри лага — на дату снимка почти половина будущего оттока выглядела живой.
2. Медиана дней молчания на дату снимка: **8 у ушедших против 4 у оставшихся**, нижний квартиль 3 против 2 — распределения перекрываются почти целиком.
3. Снятие всех четырёх признаков давности стоит **0,001 Gini** (0,624 → 0,623). Одна давность в одиночку даёт 0,425. Для сравнения, на версии таргета без лага она давала 0,724 из 0,846 — тогда модель действительно стояла на ней.

Порог тишины 60 дней выбран по кривой реактивации, лаг 30 дней — это время на скоринг, решение и контакт. Без лага модель предсказывала бы то, на что уже нельзя повлиять.

### Q6. Защита модели

Логистическая регрессия на 52 признаках из девяти источников, все окна строго до `t0`, разбиение по снимку, а не случайное.

**Качество заявляется диапазоном: Gini 0,62 на спокойном периоде, 0,52 на периоде с разовым внешним событием.** Обоснование — две независимые оценки на каждом периоде совпали между собой (0,520 и 0,526 на июне; 0,624 и 0,622 на сентябре), тогда как между периодами разница 0,10. Неустойчива не модель, а период.

**Почему не бустинг.** Он выиграл на одной паре снимков (0,651 против 0,624), парный бутстрэп дал интервал [+0,005; +0,049] — формально значимо. Но при смене обучающего периода регрессия впереди на обоих проверочных снимках, а Gini бустинга на обучении 0,960 показывает запоминание. Значимость на одной выборке и воспроизводимость — разные вещи.

**Отсутствие утечки** проверено в разделах 5 и 10.1: таблицы с будущим исключены, признаки с нулевой дисперсией в обучении отсеяны (12 из 64, включая все тарифные), разбиение по времени, пересечение клиентов между снимками проверено отдельно.

**Метрики:** ROC-AUC 0,812, Gini 0,624, KS 0,472, PR-AUC 0,366 при базовом уровне 0,054. На 5% списка Precision 0,401 — в 7,4 раза выше случайного отбора. Порог 0,5 не используется: при доле оттока 5,4% он даёт Precision 0,139. Рабочий порог задаёт экономика.